# RSNA Phase C: report supervision and validation audit

Import this complete generated notebook, attach only the official competition input, select CPU / no accelerator and Internet Off, then Run All from a clean session. It uses embedded standard-library sources; no repository imports, installation, credentials, dataset copies, DICOM decoding, or training occur.

Execution order: verify embedded source hashes; create one validated configuration; run CSV preflight; aggregate report quality; extract with the rule version and content hash frozen before evaluation; evaluate tri-state candidates against observed labels only; assess validation feasibility; write compact aggregate JSON. The ordered Phase C pipeline is implemented in the embedded run_phase_c.

An unmentioned or ambiguous condition remains UNKNOWN. Unknown source labels stay masked; candidate report labels are separate from observed labels. Agreement is exploratory and is not model performance or independent validation. Patient grouping and gold provenance require separate evidence. No raw report text, report snippets, study identifiers, or per-study candidate tables are printed or saved. Blocked and unassessed checks remain explicit; split approval and training readiness remain false. Only versioned aggregate JSON is saved under the configured output-size limit. Rebuild from project configuration to change limits; do not append configuration cells.


In [ ]:
# Reviewed source bundle: no filesystem extraction or network access.
import hashlib
import importlib.abc
import importlib.util
import json
import sys

_RSNA_BUNDLE = json.loads('{"sha256":{"__init__.py":"f7c27acdb5e30e9734b6fbb6d55809ec8bcfef85c15cbac858331d23941b994e","audit.py":"13bcb1acc910af81000805e1e32d139475fc7b97cbdcf5cd4d8d6320731cfbd5","cli.py":"64f2631094bcc8aab879b2ce4471731aa3fd7d39add1bbea5f019221c73ceff6","config.py":"6dbb61af824770c3ac0dca64636762f14f9cc71e3c3bd0193b200211996347cd","contracts.py":"34d3892dd8fe63dde0f5494815240b008082406b94e6fff0d16528b45c3fda05","inputs.py":"48fa4fe5c0c5f183e6975113cfd889ae2c8c9139e9b98b944dc3708e362f5d8e","metrics.py":"4121fb00bd7e2666b3f0885d32d1430d7e102d9ee8d70d4f7bf8d27dbea2f774","phase_c.py":"7bda9faef1ffbcbe4d744778f2cbc24650581cbb9dccc3ab7a05a7ea82eaa0a8","phase_c_config.py":"2fc2ec6ae9b252764bb161e9e474ea3aa6a869c6bfda39701f86622efed71cec","phase_c_validation.py":"473ed8538b3347ca38aa0bc00df0a0a417ca7b438157c782ccdedceaa774894e","report_agreement.py":"6e314f4ea4829bfb96bc494bea2a7d2f0bfd18f76ed6ae422b7a68cbb9460740","report_quality.py":"23cdd0c4e6e9e831357f17169fe2525c9056fa36601db5846fbe00767e4fe4fb","report_rules.py":"c51ab2725e7152dea427bb717c37c0d34df2529bc9ddec2ff3febbf55999536b","submission.py":"9dc8941373de54bfa2ebe037e485d0060baea653bec6c981aa6a447c7673d5b0","validation_support.py":"c77c859dc025f8b0ac0434b79a32f13e6764eb61033385880bde7d03941c4ebb"},"sources":{"__init__":"\\"\\"\\"Read-only audit software; DICOM support and model performance require evidence.\\"\\"\\"\\r\\n\\r\\n__version__ = \\"0.1.0\\"\\r\\n","audit":"\\"\\"\\"Initial preflight contracts only; never a Phase B or training approval.\\"\\"\\"\\r\\n\\r\\nfrom datetime import datetime, timezone\\r\\nimport json\\r\\nfrom pathlib import Path\\r\\nimport platform\\r\\nimport time\\r\\nimport uuid\\r\\n\\r\\nfrom .config import AuditConfig\\r\\nfrom .contracts import ContractError, TARGETS, audit_series, audit_studies\\r\\nfrom .inputs import InputError, MANIFEST_NAMES, discover_inputs, read_csv_manifest, safe_absolute\\r\\nfrom .submission import validate_submission\\r\\n\\r\\n\\r\\ndef _summarize(contract: dict) -> dict:\\r\\n    # Keep outputs compact; no full identifier tables, report text, or pixels.\\r\\n    return {key: value for key, value in contract.items() if key not in (\\"study_ids\\", \\"series_ids\\")}\\r\\n\\r\\n\\r\\ndef _csv_contracts(root: str, config: AuditConfig, progress: dict) -> dict:\\r\\n    tables = {}\\r\\n    columns = {}\\r\\n    hashes = {}\\r\\n    for name in MANIFEST_NAMES:\\r\\n        progress[\\"attempted_files\\"].append(name)\\r\\n        columns[name], tables[name], hashes[name] = read_csv_manifest(Path(root) / name, config)\\r\\n        progress[\\"parsed_files\\"].append(name)\\r\\n        progress[\\"sha256\\"][name] = hashes[name]\\r\\n    train = audit_studies(tables[\\"train.csv\\"], \\"train\\")\\r\\n    test = audit_studies(tables[\\"test.csv\\"], \\"test\\")\\r\\n    train_ids, test_ids = set(train[\\"study_ids\\"]), set(test[\\"study_ids\\"])\\r\\n    overlap = train_ids & test_ids\\r\\n    if overlap:\\r\\n        raise ContractError(\\"Train/test study identifiers overlap\\")\\r\\n    train_series = audit_series(tables[\\"train_series.csv\\"], train_ids)\\r\\n    test_series = audit_series(tables[\\"test_series.csv\\"], test_ids)\\r\\n    series_overlap = set(train_series[\\"series_ids\\"]) & set(test_series[\\"series_ids\\"])\\r\\n    if series_overlap:\\r\\n        raise ContractError(\\"Train/test series identifiers overlap\\")\\r\\n    missing_train = train_ids - {row[\\"StudyInstanceUID\\"] for row in tables[\\"train_series.csv\\"]}\\r\\n    missing_test = test_ids - {row[\\"StudyInstanceUID\\"] for row in tables[\\"test_series.csv\\"]}\\r\\n    if missing_train or missing_test:\\r\\n        raise ContractError(\\"Study manifest includes studies without a series reference\\")\\r\\n    sample = []\\r\\n    for row in tables[\\"sample_submission.csv\\"]:\\r\\n        converted = dict(row)\\r\\n        for target in TARGETS:\\r\\n            if target not in converted:\\r\\n                raise ContractError(\\"Sample submission is missing target columns\\")\\r\\n            try:\\r\\n                converted[target] = float(converted[target])\\r\\n            except (TypeError, ValueError) as error:\\r\\n                raise ContractError(\\"Sample submission scores must be numeric\\") from error\\r\\n        sample.append(converted)\\r\\n    submission = validate_submission(sample, test_ids, columns=columns[\\"sample_submission.csv\\"])\\r\\n    valid = all(result[\\"status\\"] == \\"PASS\\" for result in (train, test, train_series, test_series))\\r\\n    return {\\r\\n        \\"status\\": \\"VALID\\" if valid else \\"INVALID\\", \\"columns\\": columns,\\r\\n        \\"sha256\\": hashes, \\"train\\": _summarize(train), \\"test\\": _summarize(test),\\r\\n        \\"train_series\\": _summarize(train_series), \\"test_series\\": _summarize(test_series),\\r\\n        \\"sample_submission\\": submission, \\"train_test_study_overlap_count\\": 0,\\r\\n        \\"train_test_series_overlap_count\\": 0,\\r\\n        \\"scope\\": \\"CSV contracts only; no patient, report-quality, geometry or image-integrity audit\\",\\r\\n    }\\r\\n\\r\\n\\r\\ndef run_preflight(config: AuditConfig, source_identity: dict | None = None) -> dict:\\r\\n    if not isinstance(config, AuditConfig):\\r\\n        raise TypeError(\\"run_preflight requires validated AuditConfig\\")\\r\\n    started = time.perf_counter()\\r\\n    discovery = discover_inputs(config)\\r\\n    csv_contracts = {\\"status\\": \\"NOT_REQUESTED\\"}\\r\\n    status = discovery[\\"status\\"]\\r\\n    if config.inspect_csv:\\r\\n        csv_contracts = {\\"status\\": \\"BLOCKED_BY_DISCOVERY\\"}\\r\\n        if discovery[\\"selected_root\\"]:\\r\\n            progress = {\\"attempted_files\\": [], \\"parsed_files\\": [], \\"sha256\\": {}}\\r\\n            discovery[\\"csv_read_attempted\\"] = True\\r\\n            discovery[\\"csv_read_completed\\"] = False\\r\\n            discovery[\\"csv_preflight_completed\\"] = False\\r\\n            try:\\r\\n                csv_contracts = _csv_contracts(discovery[\\"selected_root\\"], config, progress)\\r\\n                status = \\"CONTRACTS_VALID\\" if csv_contracts[\\"status\\"] == \\"VALID\\" else \\"CONTRACTS_INVALID\\"\\r\\n                discovery[\\"csv_preflight_completed\\"] = True\\r\\n            except (InputError, ContractError, OSError) as error:\\r\\n                # Exceptions contain only static contract descriptions; no raw reports.\\r\\n                csv_contracts = {\\"status\\": \\"INVALID\\", \\"error_type\\": type(error).__name__,\\r\\n                                 \\"reason\\": str(error) if isinstance(error, (InputError, ContractError)) else \\"Filesystem read failed\\"}\\r\\n                status = \\"CONTRACTS_INVALID\\"\\r\\n            discovery[\\"csv_read_completed\\"] = len(progress[\\"parsed_files\\"]) == len(MANIFEST_NAMES)\\r\\n            # A parse failure may still have consumed bytes. None means unknown,\\r\\n            # while successful parsed files establish that contents were read.\\r\\n            discovery[\\"csv_contents_read\\"] = True if progress[\\"parsed_files\\"] else None\\r\\n            discovery[\\"csv_read_progress\\"] = progress\\r\\n    return {\\r\\n        \\"schema\\": \\"rsna-knee-preflight-v1\\", \\"phase\\": \\"INITIAL_SCAFFOLD\\", \\"status\\": status,\\r\\n        \\"created_utc\\": datetime.now(timezone.utc).isoformat(), \\"configuration\\": config.to_dict(),\\r\\n        \\"source_identity\\": source_identity or {\\"status\\": \\"NOT_SUPPLIED\\"},\\r\\n        \\"runtime\\": {\\"python\\": platform.python_version(), \\"platform\\": platform.system(), \\"gpu_used\\": False},\\r\\n        \\"discovery\\": discovery, \\"csv_contracts\\": csv_contracts,\\r\\n        \\"phase_b_status\\": \\"NOT_RUN\\", \\"training_ready\\": False, \\"phase1_pass\\": False,\\r\\n        \\"gold_resource\\": {\\"status\\": \\"NOT_ESTABLISHED\\", \\"observed_study_count\\": None},\\r\\n        \\"model_metrics\\": None,\\r\\n        \\"unassessed\\": [\\"patient grouping\\", \\"report quality and languages\\", \\"DICOM UID inventory\\",\\r\\n                       \\"physical geometry\\", \\"transfer syntaxes and decoding\\", \\"gold provenance\\",\\r\\n                       \\"validation split viability\\", \\"Kaggle account access\\", \\"model and inference runtime\\"],\\r\\n        \\"duration_before_report_write_seconds\\": time.perf_counter() - started,\\r\\n    }\\r\\n\\r\\n\\r\\ndef write_report(report: dict, config: AuditConfig) -> Path:\\r\\n    output = safe_absolute(config.output_dir)\\r\\n    source = safe_absolute(config.input_base)\\r\\n    if output.is_relative_to(source):\\r\\n        raise InputError(\\"Report destination must be outside input_base\\")\\r\\n    payload = json.dumps(report, indent=2, sort_keys=True, allow_nan=False) + \\"\\\\n\\"\\r\\n    if len(payload.encode(\\"utf-8\\")) > config.max_report_bytes:\\r\\n        raise InputError(\\"REPORT_SIZE_LIMIT_EXCEEDED\\")\\r\\n    output.mkdir(parents=True, exist_ok=True)\\r\\n    stamp = datetime.now(timezone.utc).strftime(\\"%Y%m%dT%H%M%S%fZ\\")\\r\\n    path = output / f\\"preflight_{stamp}_{uuid.uuid4().hex[:8]}.json\\"\\r\\n    with path.open(\\"x\\", encoding=\\"utf-8\\", newline=\\"\\\\n\\") as stream:\\r\\n        stream.write(payload)\\r\\n    return path\\r\\n","cli":"\\"\\"\\"Local entry point with no authenticated services or training commands.\\"\\"\\"\\r\\n\\r\\nimport argparse\\r\\nimport hashlib\\r\\nimport json\\r\\nfrom pathlib import Path\\r\\nimport sys\\r\\n\\r\\nfrom .audit import run_preflight, write_report\\r\\nfrom .config import AuditConfig, ConfigError, load_config\\r\\nfrom .inputs import InputError\\r\\n\\r\\n\\r\\ndef _source_identity() -> dict:\\r\\n    package = Path(__file__).parent\\r\\n    names = (\\"__init__\\", \\"config\\", \\"inputs\\", \\"contracts\\", \\"metrics\\", \\"submission\\", \\"audit\\", \\"cli\\")\\r\\n    return {\\"modules\\": {name: hashlib.sha256((package / (name + \\".py\\")).read_bytes()).hexdigest() for name in names}}\\r\\n\\r\\n\\r\\ndef main(argv=None) -> int:\\r\\n    parser = argparse.ArgumentParser(description=\\"Read-only RSNA input/CSV preflight; no Phase B execution or training\\")\\r\\n    parser.add_argument(\\"--config\\", type=Path)\\r\\n    parser.add_argument(\\"--input-base\\")\\r\\n    parser.add_argument(\\"--data-root\\")\\r\\n    parser.add_argument(\\"--output-dir\\")\\r\\n    parser.add_argument(\\"--inspect-csv\\", action=\\"store_true\\", default=None,\\r\\n                        help=\\"Check bounded CSV contracts; source data is not modified\\")\\r\\n    args = parser.parse_args(argv)\\r\\n    try:\\r\\n        config = load_config(args.config) if args.config else AuditConfig()\\r\\n        overrides = {name: getattr(args, name) for name in (\\"input_base\\", \\"data_root\\", \\"output_dir\\", \\"inspect_csv\\") if getattr(args, name) is not None}\\r\\n        config = config.with_overrides(**overrides)\\r\\n        report = run_preflight(config, source_identity=_source_identity())\\r\\n        path = write_report(report, config)\\r\\n    except (ConfigError, InputError, OSError, ValueError) as error:\\r\\n        message = str(error) if isinstance(error, (ConfigError, InputError)) else \\"Preflight failed; check configuration and filesystem permissions\\"\\r\\n        print(json.dumps({\\"status\\": \\"ERROR\\", \\"error_type\\": type(error).__name__, \\"reason\\": message}), file=sys.stderr)\\r\\n        return 2\\r\\n    print(json.dumps({\\"status\\": report[\\"status\\"], \\"report_path\\": str(path),\\r\\n                      \\"phase_b_status\\": report[\\"phase_b_status\\"], \\"training_ready\\": report[\\"training_ready\\"]}, indent=2))\\r\\n    return 0 if report[\\"status\\"] in (\\"MANIFESTS_LOCATED\\", \\"CONTRACTS_VALID\\") else 2\\r\\n","config":"\\"\\"\\"Small immutable configuration; no credential or environment discovery.\\"\\"\\"\\r\\n\\r\\nfrom dataclasses import asdict, dataclass, fields, replace\\r\\nimport json\\r\\nimport os\\r\\nfrom pathlib import Path\\r\\nimport stat\\r\\nfrom typing import Mapping\\r\\n\\r\\n\\r\\nclass ConfigError(ValueError):\\r\\n    \\"\\"\\"Configuration cannot safely represent the requested preflight.\\"\\"\\"\\r\\n\\r\\n\\r\\n@dataclass(frozen=True)\\r\\nclass AuditConfig:\\r\\n    input_base: str = \\"data\\"\\r\\n    output_dir: str = \\"artifacts\\"\\r\\n    data_root: str | None = None\\r\\n    inspect_csv: bool = False\\r\\n    max_depth: int = 2\\r\\n    max_directories: int = 128\\r\\n    max_entries: int = 2048\\r\\n    max_file_bytes: int = 8 * 1024 * 1024\\r\\n    max_csv_rows: int = 100000\\r\\n    max_report_bytes: int = 1024 * 1024\\r\\n\\r\\n    def __post_init__(self):\\r\\n        for name in (\\"input_base\\", \\"output_dir\\", \\"data_root\\"):\\r\\n            value = getattr(self, name)\\r\\n            if value is None and name == \\"data_root\\":\\r\\n                continue\\r\\n            if not isinstance(value, str) or not value.strip() or value != value.strip():\\r\\n                raise ConfigError(f\\"{name} must be a nonempty unpadded path string\\")\\r\\n            if any(ord(character) < 32 for character in value):\\r\\n                raise ConfigError(f\\"{name} contains a control character\\")\\r\\n        if type(self.inspect_csv) is not bool:\\r\\n            raise ConfigError(\\"inspect_csv must be a JSON boolean\\")\\r\\n        limits = {\\r\\n            \\"max_depth\\": (0, 4), \\"max_directories\\": (1, 1024),\\r\\n            \\"max_entries\\": (1, 65536), \\"max_file_bytes\\": (1, 64 * 1024 * 1024),\\r\\n            \\"max_csv_rows\\": (1, 1000000), \\"max_report_bytes\\": (1, 4 * 1024 * 1024),\\r\\n        }\\r\\n        for name, (lower, upper) in limits.items():\\r\\n            value = getattr(self, name)\\r\\n            if type(value) is not int or not lower <= value <= upper:\\r\\n                raise ConfigError(f\\"{name} must be an integer in [{lower}, {upper}]\\")\\r\\n\\r\\n    @classmethod\\r\\n    def from_dict(cls, values: Mapping):\\r\\n        if not isinstance(values, Mapping):\\r\\n            raise ConfigError(\\"Configuration must be a JSON object\\")\\r\\n        unknown = set(values) - {field.name for field in fields(cls)}\\r\\n        if unknown:\\r\\n            raise ConfigError(\\"Unknown configuration fields: \\" + \\", \\".join(sorted(str(key) for key in unknown)))\\r\\n        return cls(**dict(values))\\r\\n\\r\\n    def to_dict(self) -> dict:\\r\\n        return asdict(self)\\r\\n\\r\\n    def with_overrides(self, **values):\\r\\n        try:\\r\\n            return replace(self, **values)\\r\\n        except TypeError as error:\\r\\n            raise ConfigError(\\"Invalid configuration override\\") from error\\r\\n\\r\\n\\r\\ndef _read_config_values(path: str | Path) -> dict:\\r\\n    \\"\\"\\"Shared bounded JSON reader; never discover environment or credentials.\\"\\"\\"\\r\\n    def unique_pairs(pairs):\\r\\n        values = {}\\r\\n        for key, value in pairs:\\r\\n            if key in values:\\r\\n                raise ConfigError(f\\"Duplicate configuration field: {key}\\")\\r\\n            values[key] = value\\r\\n        return values\\r\\n\\r\\n    source = Path(os.path.abspath(path))\\r\\n    protected = {\\".git\\", \\".aws\\", \\".kaggle\\", \\".codex\\", \\"credentials\\", \\"access_token\\", \\"kaggle.json\\"}\\r\\n    if source.suffix.casefold() != \\".json\\" or any(part.casefold() in protected or part.casefold().startswith(\\".env\\") for part in source.parts):\\r\\n        raise ConfigError(\\"Configuration must be a project JSON file outside protected paths\\")\\r\\n    for ancestor in (source, *source.parents):\\r\\n        try:\\r\\n            info = ancestor.lstat()\\r\\n        except FileNotFoundError:\\r\\n            continue\\r\\n        if stat.S_ISLNK(info.st_mode) or getattr(info, \\"st_file_attributes\\", 0) & 0x400:\\r\\n            raise ConfigError(\\"Configuration cannot use symbolic links or reparse paths\\")\\r\\n    if not stat.S_ISREG(source.lstat().st_mode):\\r\\n        raise ConfigError(\\"Configuration must be a regular project JSON file\\")\\r\\n    with source.open(\\"rb\\") as stream:\\r\\n        raw = stream.read(64 * 1024 + 1)\\r\\n    if len(raw) > 64 * 1024:\\r\\n        raise ConfigError(\\"Configuration exceeds the 64 KiB budget\\")\\r\\n    try:\\r\\n        values = json.loads(raw.decode(\\"utf-8-sig\\"), object_pairs_hook=unique_pairs)\\r\\n    except (json.JSONDecodeError, UnicodeError) as error:\\r\\n        raise ConfigError(\\"Configuration must be valid UTF-8 JSON\\") from error\\r\\n    return values\\r\\n\\r\\n\\r\\ndef load_config(path: str | Path) -> AuditConfig:\\r\\n    return AuditConfig.from_dict(_read_config_values(path))\\r\\n","contracts":"\\"\\"\\"Small, explicit contracts for competition identifiers and supervision.\\r\\n\\r\\nUnknown labels never become negatives. These functions inspect tabular values;\\r\\nthey neither decode images nor infer labels from reports.\\r\\n\\"\\"\\"\\r\\n\\r\\nfrom __future__ import annotations\\r\\n\\r\\nimport math\\r\\nimport unicodedata\\r\\nfrom collections import Counter\\r\\nfrom collections.abc import Mapping\\r\\nfrom decimal import Decimal\\r\\nfrom numbers import Real\\r\\nfrom typing import Any\\r\\n\\r\\nTARGETS = (\\r\\n    \\"ACL\\", \\"MCL\\", \\"Medial Meniscus\\", \\"Lateral Meniscus\\", \\"Medial OA\\",\\r\\n    \\"Lateral OA\\", \\"PF OA\\", \\"Effusion\\", \\"Synovitis\\", \\"Baker\'s\\",\\r\\n    \\"Contusion\\", \\"Fracture\\",\\r\\n)\\r\\nSTUDY_ID = \\"StudyInstanceUID\\"\\r\\nSERIES_ID = \\"SeriesInstanceUID\\"\\r\\nSERIES_COLUMNS = (\\r\\n    STUDY_ID, SERIES_ID, \\"Anatomical_Plane\\", \\"Fluid_Sensitive\\", \\"Fat_Suppression\\",\\r\\n)\\r\\n\\r\\n\\r\\nclass ContractError(ValueError):\\r\\n    \\"\\"\\"A table violates a declared data contract.\\"\\"\\"\\r\\n\\r\\n\\r\\ndef validate_identifier(value: str) -> str:\\r\\n    \\"\\"\\"Return an unchanged safe identifier without imposing a numeric UID format.\\"\\"\\"\\r\\n    if not isinstance(value, str) or not value or value != value.strip():\\r\\n        raise ContractError(\\"identifier must be a nonempty, unpadded string\\")\\r\\n    if value in {\\".\\", \\"..\\"} or any(char in value for char in \\"/\\\\\\\\:\\"):\\r\\n        raise ContractError(\\"identifier must be a single safe path component\\")\\r\\n    if any(unicodedata.category(char) in {\\"Cc\\", \\"Cf\\"} for char in value):\\r\\n        raise ContractError(\\"identifier contains a control character\\")\\r\\n    return value\\r\\n\\r\\n\\r\\ndef _numeric(value: Any) -> bool:\\r\\n    return not isinstance(value, bool) and isinstance(value, (Real, Decimal))\\r\\n\\r\\n\\r\\ndef parse_label(value: Any) -> tuple[float | None, str]:\\r\\n    \\"\\"\\"Classify an observed binary label, an unknown cell, or an invalid cell.\\r\\n\\r\\n    Accepted text is deliberately limited to 0, 1, 0.0 and 1.0. A text token\\r\\n    such as \'NaN\' is invalid; an actual floating NaN represents a missing cell.\\r\\n    \\"\\"\\"\\r\\n    if value is None:\\r\\n        return None, \\"unknown\\"\\r\\n    if isinstance(value, str):\\r\\n        token = value.strip()\\r\\n        if not token:\\r\\n            return None, \\"unknown\\"\\r\\n        if token in {\\"0\\", \\"0.0\\", \\"1\\", \\"1.0\\"}:\\r\\n            return float(token), \\"observed\\"\\r\\n        return None, \\"invalid\\"\\r\\n    if isinstance(value, Real) and not isinstance(value, bool):\\r\\n        try:\\r\\n            if math.isnan(float(value)):\\r\\n                return None, \\"unknown\\"\\r\\n        except (OverflowError, ValueError):\\r\\n            return None, \\"invalid\\"\\r\\n    if _numeric(value):\\r\\n        try:\\r\\n            numeric = float(value)\\r\\n        except (OverflowError, ValueError):\\r\\n            return None, \\"invalid\\"\\r\\n        if math.isfinite(numeric) and numeric in (0.0, 1.0):\\r\\n            return numeric, \\"observed\\"\\r\\n    return None, \\"invalid\\"\\r\\n\\r\\n\\r\\ndef _required(row: Mapping[str, Any], required: tuple[str, ...], index: int) -> None:\\r\\n    if not isinstance(row, Mapping):\\r\\n        raise ContractError(f\\"row {index} must be a mapping\\")\\r\\n    missing = [key for key in required if key not in row]\\r\\n    if missing:\\r\\n        raise ContractError(f\\"row {index} is missing required columns: {\', \'.join(missing)}\\")\\r\\n\\r\\n\\r\\ndef label_vector(row: Mapping[str, Any]) -> dict[str, Any]:\\r\\n    \\"\\"\\"Return twelve ordered values and an explicit observation mask.\\r\\n\\r\\n    Unknown and invalid cells retain None; the state list records the\\r\\n    distinction. Only verified binary observations receive a True mask.\\r\\n    \\"\\"\\"\\r\\n    _required(row, TARGETS, 0)\\r\\n    parsed = [parse_label(row[target]) for target in TARGETS]\\r\\n    return {\\r\\n        \\"targets\\": list(TARGETS),\\r\\n        \\"values\\": [value for value, _ in parsed],\\r\\n        \\"observed_mask\\": [state == \\"observed\\" for _, state in parsed],\\r\\n        \\"states\\": [state for _, state in parsed],\\r\\n    }\\r\\n\\r\\n\\r\\ndef audit_studies(rows: list[dict[str, Any]], split: str) -> dict[str, Any]:\\r\\n    \\"\\"\\"Audit study identities and, for training, labels and report coverage.\\"\\"\\"\\r\\n    if split not in {\\"train\\", \\"test\\"}:\\r\\n        raise ContractError(\\"split must be train or test\\")\\r\\n    if not rows:\\r\\n        raise ContractError(\\"study table must contain at least one row\\")\\r\\n    required = (STUDY_ID, *TARGETS, \\"Report\\") if split == \\"train\\" else (STUDY_ID,)\\r\\n    study_ids: list[str] = []\\r\\n    seen: set[str] = set()\\r\\n    statistics = {\\r\\n        target: {\\"observed\\": 0, \\"positive\\": 0, \\"negative\\": 0, \\"unknown\\": 0, \\"invalid\\": 0}\\r\\n        for target in TARGETS\\r\\n    }\\r\\n    label_states: list[dict[str, tuple[float | None, str]]] = []\\r\\n    report_summary = {\\"present\\": 0, \\"empty_or_missing\\": 0, \\"invalid\\": 0}\\r\\n    for index, row in enumerate(rows):\\r\\n        _required(row, required, index)\\r\\n        uid = validate_identifier(row[STUDY_ID])\\r\\n        if uid in seen:\\r\\n            raise ContractError(f\\"duplicate study identifier at row {index}\\")\\r\\n        seen.add(uid)\\r\\n        study_ids.append(uid)\\r\\n        if split == \\"test\\":\\r\\n            continue\\r\\n        states = {target: parse_label(row[target]) for target in TARGETS}\\r\\n        label_states.append(states)\\r\\n        for target, (label, state) in states.items():\\r\\n            statistics[target][state] += 1\\r\\n            if state == \\"observed\\":\\r\\n                statistics[target][\\"positive\\" if label == 1.0 else \\"negative\\"] += 1\\r\\n        report = row[\\"Report\\"]\\r\\n        if report is None or (isinstance(report, str) and not report.strip()):\\r\\n            report_summary[\\"empty_or_missing\\"] += 1\\r\\n        elif isinstance(report, str):\\r\\n            report_summary[\\"present\\"] += 1\\r\\n        else:\\r\\n            report_summary[\\"invalid\\"] += 1\\r\\n    result: dict[str, Any] = {\\r\\n        \\"status\\": \\"PASS\\", \\"split\\": split, \\"row_count\\": len(rows),\\r\\n        \\"unique_study_count\\": len(seen), \\"study_ids\\": study_ids,\\r\\n    }\\r\\n    if split == \\"test\\":\\r\\n        return result\\r\\n    for stats in statistics.values():\\r\\n        observed = stats[\\"observed\\"]\\r\\n        stats[\\"prevalence\\"] = stats[\\"positive\\"] / observed if observed else None\\r\\n        stats[\\"observation_coverage\\"] = observed / len(rows)\\r\\n    cooccurrence = []\\r\\n    for first_index, first in enumerate(TARGETS):\\r\\n        for second in TARGETS[first_index + 1:]:\\r\\n            jointly_observed = both_positive = 0\\r\\n            for states in label_states:\\r\\n                first_label, first_state = states[first]\\r\\n                second_label, second_state = states[second]\\r\\n                if first_state == second_state == \\"observed\\":\\r\\n                    jointly_observed += 1\\r\\n                    both_positive += first_label == second_label == 1.0\\r\\n            cooccurrence.append({\\r\\n                \\"targets\\": [first, second], \\"jointly_observed\\": jointly_observed,\\r\\n                \\"both_positive\\": both_positive,\\r\\n                \\"both_positive_fraction\\": both_positive / jointly_observed if jointly_observed else None,\\r\\n            })\\r\\n    result.update({\\r\\n        \\"target_statistics\\": statistics, \\"report_summary\\": report_summary,\\r\\n        \\"observation_coverage\\": sum(stats[\\"observed\\"] for stats in statistics.values()) / (len(rows) * len(TARGETS)),\\r\\n        \\"jointly_observed_cooccurrence\\": cooccurrence,\\r\\n    })\\r\\n    if any(stats[\\"invalid\\"] for stats in statistics.values()) or report_summary[\\"invalid\\"]:\\r\\n        result[\\"status\\"] = \\"INVALID\\"\\r\\n    return result\\r\\n\\r\\n\\r\\ndef _metadata_state(value: Any) -> str:\\r\\n    if value is None or (isinstance(value, str) and not value.strip()):\\r\\n        return \\"unknown\\"\\r\\n    if isinstance(value, float):\\r\\n        if math.isnan(float(value)):\\r\\n            return \\"unknown\\"\\r\\n        if not math.isfinite(float(value)):\\r\\n            return \\"invalid\\"\\r\\n    if isinstance(value, (str, bool, int, float)):\\r\\n        return \\"observed\\"\\r\\n    return \\"invalid\\"\\r\\n\\r\\n\\r\\ndef audit_series(rows: list[dict[str, Any]], study_ids: set[str]) -> dict[str, Any]:\\r\\n    \\"\\"\\"Check series uniqueness, study references and exact metadata coverage.\\r\\n\\r\\n    Category vocabularies must be established from real data. No plane or flag\\r\\n    value is coerced here; typed value counts retain the observed representation.\\r\\n    \\"\\"\\"\\r\\n    if not rows:\\r\\n        raise ContractError(\\"series table must contain at least one row\\")\\r\\n    validated_studies = {validate_identifier(uid) for uid in study_ids}\\r\\n    seen: set[str] = set()\\r\\n    referenced: set[str] = set()\\r\\n    metadata_fields = SERIES_COLUMNS[2:]\\r\\n    statistics = {field: {\\"observed\\": 0, \\"unknown\\": 0, \\"invalid\\": 0} for field in metadata_fields}\\r\\n    counts: dict[str, Counter[tuple[str, Any]]] = {field: Counter() for field in metadata_fields}\\r\\n    series_ids: list[str] = []\\r\\n    for index, row in enumerate(rows):\\r\\n        _required(row, SERIES_COLUMNS, index)\\r\\n        study = validate_identifier(row[STUDY_ID])\\r\\n        series = validate_identifier(row[SERIES_ID])\\r\\n        if study not in validated_studies:\\r\\n            raise ContractError(f\\"series row {index} references an absent study\\")\\r\\n        if series in seen:\\r\\n            raise ContractError(f\\"duplicate series identifier at row {index}\\")\\r\\n        seen.add(series)\\r\\n        series_ids.append(series)\\r\\n        referenced.add(study)\\r\\n        for field in metadata_fields:\\r\\n            value = row[field]\\r\\n            state = _metadata_state(value)\\r\\n            statistics[field][state] += 1\\r\\n            if state == \\"observed\\":\\r\\n                counts[field][(type(value).__name__, value)] += 1\\r\\n    for field in metadata_fields:\\r\\n        statistics[field][\\"value_counts\\"] = [\\r\\n            {\\"type\\": value_type, \\"value\\": value, \\"count\\": count}\\r\\n            for (value_type, value), count in sorted(counts[field].items(), key=lambda item: (item[0][0], str(item[0][1])))\\r\\n        ]\\r\\n    return {\\r\\n        \\"status\\": \\"INVALID\\" if any(stats[\\"invalid\\"] for stats in statistics.values()) else \\"PASS\\",\\r\\n        \\"row_count\\": len(rows), \\"unique_series_count\\": len(seen),\\r\\n        \\"study_count_with_series\\": len(referenced), \\"study_count_without_series\\": len(validated_studies - referenced),\\r\\n        \\"series_ids\\": series_ids, \\"metadata_statistics\\": statistics,\\r\\n    }\\r\\n","inputs":"\\"\\"\\"Bounded read-only discovery and CSV reads with explicit incomplete states.\\"\\"\\"\\r\\n\\r\\nimport csv\\r\\nfrom datetime import datetime, timezone\\r\\nimport hashlib\\r\\nimport io\\r\\nimport os\\r\\nfrom pathlib import Path\\r\\nimport stat\\r\\n\\r\\nfrom .config import AuditConfig\\r\\n\\r\\n\\r\\nMANIFEST_NAMES = (\\"train.csv\\", \\"train_series.csv\\", \\"test.csv\\", \\"test_series.csv\\", \\"sample_submission.csv\\")\\r\\nIMAGE_DIRECTORIES = {\\"train_series\\", \\"test_series\\", \\"train_images\\", \\"test_images\\"}\\r\\nPROTECTED_COMPONENTS = {\\".git\\", \\".aws\\", \\".kaggle\\", \\".codex\\", \\".venv\\", \\"venv\\", \\"node_modules\\", \\"credentials\\", \\"access_token\\", \\"kaggle.json\\"}\\r\\n\\r\\n\\r\\nclass InputError(ValueError):\\r\\n    \\"\\"\\"Input contracts or safe-path requirements were not satisfied.\\"\\"\\"\\r\\n\\r\\n\\r\\ndef safe_absolute(path: str | Path) -> Path:\\r\\n    \\"\\"\\"Reject links/reparse ancestors rather than silently resolving through them.\\"\\"\\"\\r\\n    absolute = Path(os.path.abspath(path))\\r\\n    if any(part.casefold() in PROTECTED_COMPONENTS or part.casefold().startswith(\\".env\\") for part in absolute.parts):\\r\\n        raise InputError(\\"Protected directory is not a permitted input or output path\\")\\r\\n    for ancestor in (absolute, *absolute.parents):\\r\\n        try:\\r\\n            info = ancestor.lstat()\\r\\n        except FileNotFoundError:\\r\\n            continue\\r\\n        if stat.S_ISLNK(info.st_mode) or getattr(info, \\"st_file_attributes\\", 0) & 0x400:\\r\\n            raise InputError(\\"Symbolic links and reparse points are not permitted\\")\\r\\n    return absolute\\r\\n\\r\\n\\r\\ndef discover_inputs(config: AuditConfig) -> dict:\\r\\n    result = {\\r\\n        \\"status\\": \\"INPUTS_NOT_LOCATED\\", \\"input_base\\": None, \\"selected_root\\": None,\\r\\n        \\"directories_checked\\": 0, \\"entries_checked\\": 0, \\"scan_complete\\": True,\\r\\n        \\"limit_reached\\": False, \\"errors\\": [], \\"error_count\\": 0,\\r\\n        \\"skipped_links\\": [], \\"skipped_link_count\\": 0, \\"candidates\\": [],\\r\\n        \\"csv_contents_read\\": False, \\"dicom_contents_read\\": False,\\r\\n        \\"dataset_identity_verified\\": False,\\r\\n    }\\r\\n\\r\\n    def issue(path, error):\\r\\n        result[\\"error_count\\"] += 1\\r\\n        if len(result[\\"errors\\"]) < 32:\\r\\n            result[\\"errors\\"].append({\\"path\\": str(path), \\"type\\": type(error).__name__, \\"errno\\": getattr(error, \\"errno\\", None)})\\r\\n\\r\\n    try:\\r\\n        base = safe_absolute(config.input_base)\\r\\n        result[\\"input_base\\"] = str(base)\\r\\n        start = safe_absolute(config.data_root) if config.data_root else base\\r\\n        if not start.is_relative_to(base):\\r\\n            raise InputError(\\"Explicit data_root must be within input_base\\")\\r\\n        info = start.lstat()\\r\\n        if not stat.S_ISDIR(info.st_mode):\\r\\n            raise InputError(\\"Input root is not a directory\\")\\r\\n    except FileNotFoundError:\\r\\n        result[\\"status\\"] = \\"INPUT_ROOT_MISSING\\" if config.data_root else \\"INPUT_BASE_MISSING\\"\\r\\n        return result\\r\\n    except (OSError, InputError) as error:\\r\\n        issue(config.input_base, error)\\r\\n        result.update(status=\\"INPUT_PATH_INVALID\\", scan_complete=False)\\r\\n        return result\\r\\n\\r\\n    def checked_stat(path):\\r\\n        try:\\r\\n            info = path.lstat()\\r\\n            if stat.S_ISLNK(info.st_mode) or getattr(info, \\"st_file_attributes\\", 0) & 0x400:\\r\\n                result[\\"skipped_link_count\\"] += 1\\r\\n                if len(result[\\"skipped_links\\"]) < 32:\\r\\n                    result[\\"skipped_links\\"].append(str(path))\\r\\n                return None\\r\\n            if not path.resolve().is_relative_to(base):\\r\\n                raise InputError(\\"Discovered path escaped input_base\\")\\r\\n            return info\\r\\n        except FileNotFoundError:\\r\\n            return None\\r\\n        except (OSError, InputError) as error:\\r\\n            issue(path, error)\\r\\n            return None\\r\\n\\r\\n    pending = [(start, 0)]\\r\\n    cursor = 0\\r\\n    while cursor < len(pending) and result[\\"directories_checked\\"] < config.max_directories:\\r\\n        directory, depth = pending[cursor]\\r\\n        cursor += 1\\r\\n        result[\\"directories_checked\\"] += 1\\r\\n        files = []\\r\\n        for name in MANIFEST_NAMES:\\r\\n            path = directory / name\\r\\n            info = checked_stat(path)\\r\\n            if info is not None and stat.S_ISREG(info.st_mode):\\r\\n                files.append({\\"name\\": name, \\"path\\": str(path), \\"size_bytes\\": info.st_size,\\r\\n                              \\"modified_utc\\": datetime.fromtimestamp(info.st_mtime, timezone.utc).isoformat(), \\"sha256\\": None})\\r\\n        if files:\\r\\n            present = {item[\\"name\\"] for item in files}\\r\\n            images = {}\\r\\n            for name in sorted(IMAGE_DIRECTORIES):\\r\\n                path = directory / name\\r\\n                info = checked_stat(path)\\r\\n                if info is not None and stat.S_ISDIR(info.st_mode):\\r\\n                    images[name] = str(path)\\r\\n            result[\\"candidates\\"].append({\\"root\\": str(directory), \\"files\\": files,\\r\\n                                         \\"missing\\": [name for name in MANIFEST_NAMES if name not in present],\\r\\n                                         \\"image_directory_candidates\\": images, \\"dicom_file_count\\": None})\\r\\n        if config.data_root is not None or depth >= config.max_depth:\\r\\n            continue\\r\\n        try:\\r\\n            with os.scandir(directory) as entries:\\r\\n                for entry in entries:\\r\\n                    if result[\\"entries_checked\\"] >= config.max_entries:\\r\\n                        result[\\"limit_reached\\"] = True\\r\\n                        break\\r\\n                    result[\\"entries_checked\\"] += 1\\r\\n                    if entry.name.startswith(\\".\\") or entry.name in IMAGE_DIRECTORIES or entry.name.casefold() in PROTECTED_COMPONENTS:\\r\\n                        continue\\r\\n                    child = directory / entry.name\\r\\n                    info = checked_stat(child)\\r\\n                    if info is not None and stat.S_ISDIR(info.st_mode):\\r\\n                        pending.append((child, depth + 1))\\r\\n        except OSError as error:\\r\\n            issue(directory, error)\\r\\n        if result[\\"limit_reached\\"]:\\r\\n            break\\r\\n    result[\\"scan_complete\\"] = cursor == len(pending) and not result[\\"limit_reached\\"] and not result[\\"error_count\\"]\\r\\n    result[\\"candidates\\"].sort(key=lambda item: item[\\"root\\"])\\r\\n    complete = [item for item in result[\\"candidates\\"] if not item[\\"missing\\"]]\\r\\n    if len(complete) > 1:\\r\\n        result[\\"status\\"] = \\"AMBIGUOUS_INPUT_ROOTS\\"\\r\\n    elif not result[\\"scan_complete\\"]:\\r\\n        result[\\"status\\"] = \\"DISCOVERY_INCOMPLETE\\"\\r\\n    elif complete:\\r\\n        result.update(status=\\"MANIFESTS_LOCATED\\", selected_root=complete[0][\\"root\\"])\\r\\n    elif result[\\"candidates\\"]:\\r\\n        result[\\"status\\"] = \\"INCOMPLETE_MANIFESTS\\"\\r\\n    return result\\r\\n\\r\\n\\r\\ndef read_csv_manifest(path: str | Path, config: AuditConfig) -> tuple[list[str], list[dict], str]:\\r\\n    source = safe_absolute(path)\\r\\n    base = safe_absolute(config.input_base)\\r\\n    if not source.is_relative_to(base) or source.name not in MANIFEST_NAMES:\\r\\n        raise InputError(\\"CSV source must be a named competition manifest within input_base\\")\\r\\n    if not stat.S_ISREG(source.lstat().st_mode):\\r\\n        raise InputError(\\"CSV source must be a regular file\\")\\r\\n    with source.open(\\"rb\\") as stream:\\r\\n        if not stat.S_ISREG(os.fstat(stream.fileno()).st_mode):\\r\\n            raise InputError(\\"CSV source must be a regular file\\")\\r\\n        raw = stream.read(config.max_file_bytes + 1)\\r\\n    if len(raw) > config.max_file_bytes:\\r\\n        raise InputError(\\"CSV_FILE_LIMIT_EXCEEDED\\")\\r\\n    try:\\r\\n        reader = csv.DictReader(io.StringIO(raw.decode(\\"utf-8-sig\\"), newline=\\"\\"), strict=True)\\r\\n        columns = reader.fieldnames\\r\\n        if not columns or any(not name for name in columns) or len(set(columns)) != len(columns):\\r\\n            raise InputError(\\"CSV_HEADERS_MISSING_OR_DUPLICATED\\")\\r\\n        rows = []\\r\\n        for number, row in enumerate(reader, start=1):\\r\\n            if number > config.max_csv_rows:\\r\\n                raise InputError(\\"CSV_ROW_LIMIT_EXCEEDED\\")\\r\\n            if None in row or any(value is None for value in row.values()):\\r\\n                raise InputError(\\"CSV_ROW_WIDTH_INVALID\\")\\r\\n            rows.append(row)\\r\\n    except (UnicodeError, csv.Error) as error:\\r\\n        raise InputError(\\"CSV_ENCODING_OR_FORMAT_INVALID\\") from error\\r\\n    return columns, rows, hashlib.sha256(raw).hexdigest()\\r\\n","metrics":"\\"\\"\\"Observed-label ROC AUC with explicit support and undefined macro metrics.\\"\\"\\"\\r\\n\\r\\nfrom __future__ import annotations\\r\\n\\r\\nimport math\\r\\nfrom collections.abc import Iterable, Mapping, Sequence\\r\\nfrom typing import Any\\r\\n\\r\\nfrom .contracts import ContractError, STUDY_ID, TARGETS, _numeric, parse_label, validate_identifier\\r\\nfrom .submission import validate_submission\\r\\n\\r\\n\\r\\ndef binary_auc(labels: Iterable[Any], scores: Iterable[Any]) -> float | None:\\r\\n    \\"\\"\\"Compute rank AUC, averaging tied ranks; return None for missing classes.\\"\\"\\"\\r\\n    observed_labels = list(labels)\\r\\n    observed_scores = list(scores)\\r\\n    if len(observed_labels) != len(observed_scores):\\r\\n        raise ContractError(\\"AUC labels and scores must have identical lengths\\")\\r\\n    pairs = []\\r\\n    for label, score in zip(observed_labels, observed_scores):\\r\\n        if not _numeric(label) or parse_label(label)[1] != \\"observed\\":\\r\\n            raise ContractError(\\"AUC requires observed numeric binary labels\\")\\r\\n        if not _numeric(score):\\r\\n            raise ContractError(\\"AUC requires numeric scores\\")\\r\\n        try:\\r\\n            numeric_score = float(score)\\r\\n        except (OverflowError, ValueError) as error:\\r\\n            raise ContractError(\\"AUC requires finite scores\\") from error\\r\\n        if not math.isfinite(numeric_score):\\r\\n            raise ContractError(\\"AUC requires finite scores\\")\\r\\n        pairs.append((numeric_score, float(label)))\\r\\n    positive_count = sum(label == 1.0 for _, label in pairs)\\r\\n    negative_count = len(pairs) - positive_count\\r\\n    if not positive_count or not negative_count:\\r\\n        return None\\r\\n    pairs.sort(key=lambda pair: pair[0])\\r\\n    positive_rank_sum = 0.0\\r\\n    start = 0\\r\\n    while start < len(pairs):\\r\\n        end = start + 1\\r\\n        while end < len(pairs) and pairs[end][0] == pairs[start][0]:\\r\\n            end += 1\\r\\n        mean_rank = (start + 1 + end) / 2.0\\r\\n        positive_rank_sum += mean_rank * sum(label == 1.0 for _, label in pairs[start:end])\\r\\n        start = end\\r\\n    return (positive_rank_sum - positive_count * (positive_count + 1) / 2.0) / (positive_count * negative_count)\\r\\n\\r\\n\\r\\ndef evaluate_predictions(\\r\\n    label_rows: Sequence[Mapping[str, Any]],\\r\\n    prediction_rows: Sequence[Mapping[str, Any]],\\r\\n) -> dict[str, Any]:\\r\\n    \\"\\"\\"Match studies exactly and evaluate each target on observed labels only.\\"\\"\\"\\r\\n    if not label_rows:\\r\\n        raise ContractError(\\"reference label table must not be empty\\")\\r\\n    references: dict[str, Mapping[str, Any]] = {}\\r\\n    for index, row in enumerate(label_rows):\\r\\n        if not isinstance(row, Mapping) or not set((STUDY_ID, *TARGETS)).issubset(row):\\r\\n            raise ContractError(f\\"reference row {index} is missing required cells\\")\\r\\n        uid = validate_identifier(row[STUDY_ID])\\r\\n        if uid in references:\\r\\n            raise ContractError(f\\"duplicate reference identifier at row {index}\\")\\r\\n        references[uid] = row\\r\\n    validate_submission(prediction_rows, references)\\r\\n    predictions = {row[STUDY_ID]: row for row in prediction_rows}\\r\\n    per_target = {}\\r\\n    for target in TARGETS:\\r\\n        labels = []\\r\\n        scores = []\\r\\n        unknown_count = 0\\r\\n        for uid, row in references.items():\\r\\n            label, state = parse_label(row[target])\\r\\n            if state == \\"invalid\\":\\r\\n                raise ContractError(f\\"reference contains an invalid label for {target}\\")\\r\\n            if state == \\"unknown\\":\\r\\n                unknown_count += 1\\r\\n                continue\\r\\n            labels.append(label)\\r\\n            scores.append(predictions[uid][target])\\r\\n        positives = sum(label == 1.0 for label in labels)\\r\\n        per_target[target] = {\\r\\n            \\"observed\\": len(labels), \\"positive\\": positives,\\r\\n            \\"negative\\": len(labels) - positives, \\"unknown\\": unknown_count,\\r\\n            \\"auc\\": binary_auc(labels, scores),\\r\\n        }\\r\\n    defined = [stats[\\"auc\\"] for stats in per_target.values() if stats[\\"auc\\"] is not None]\\r\\n    complete = len(defined) == len(TARGETS)\\r\\n    return {\\r\\n        \\"status\\": \\"FULL_METRICS_AVAILABLE\\" if complete else \\"INCOMPLETE_METRICS\\",\\r\\n        \\"row_count\\": len(references), \\"per_target\\": per_target,\\r\\n        \\"macro_auc\\": sum(defined) / len(TARGETS) if complete else None,\\r\\n        \\"diagnostic_defined_target_macro_auc\\": sum(defined) / len(defined) if defined else None,\\r\\n        \\"defined_target_count\\": len(defined), \\"required_target_count\\": len(TARGETS),\\r\\n    }\\r\\n","phase_c":"\\"\\"\\"Read-only report supervision audit with frozen rules and aggregate evidence.\\"\\"\\"\\n\\nfrom collections import Counter\\nfrom datetime import datetime, timezone\\nimport json\\nfrom pathlib import Path\\nimport platform\\nimport re\\nimport time\\nimport uuid\\n\\nfrom .audit import run_preflight\\nfrom .contracts import ContractError\\nfrom .inputs import InputError, MANIFEST_NAMES, read_csv_manifest, safe_absolute\\nfrom .phase_c_config import PhaseCConfig\\nfrom .phase_c_validation import assess_validation\\nfrom .report_quality import aggregate_reports\\n\\n\\nCHECK_STATES = {\\"PASS\\", \\"FAIL\\", \\"BLOCKED\\", \\"NOT_ASSESSED\\"}\\nSAFE_SOURCE_FILES = frozenset(name + \\".py\\" for name in (\\n    \\"__init__\\", \\"config\\", \\"inputs\\", \\"contracts\\", \\"metrics\\", \\"submission\\", \\"audit\\", \\"cli\\",\\n    \\"report_quality\\", \\"validation_support\\", \\"report_rules\\", \\"report_agreement\\",\\n    \\"phase_c_config\\", \\"phase_c_validation\\", \\"phase_c\\",\\n))\\n\\n\\ndef _validated_source_identity(value):\\n    \\"\\"\\"Accept only the generated bundle schema, never arbitrary caller text.\\"\\"\\"\\n    if value is None or value == {\\"status\\": \\"NOT_SUPPLIED\\"}:\\n        return {\\"status\\": \\"NOT_SUPPLIED\\"}\\n    if (not isinstance(value, dict) or set(value) != {\\"kind\\", \\"bundle_sha256\\", \\"source_sha256\\"}\\n            or value.get(\\"kind\\") != \\"embedded-notebook\\" or not isinstance(value.get(\\"source_sha256\\"), dict)\\n            or set(value[\\"source_sha256\\"]) != SAFE_SOURCE_FILES):\\n        raise InputError(\\"Invalid Phase C source identity schema\\")\\n    hashes = [value[\\"bundle_sha256\\"], *value[\\"source_sha256\\"].values()]\\n    if any(not isinstance(digest, str) or re.fullmatch(r\\"[0-9a-f]{64}\\", digest) is None for digest in hashes):\\n        raise InputError(\\"Invalid Phase C source checksum\\")\\n    return {\\"kind\\": \\"embedded-notebook\\", \\"bundle_sha256\\": value[\\"bundle_sha256\\"],\\n            \\"source_sha256\\": dict(value[\\"source_sha256\\"])}\\n\\n\\ndef _csv_summary(preflight):\\n    contracts = preflight[\\"csv_contracts\\"]\\n    discovery = preflight[\\"discovery\\"]\\n    summary = {\\"status\\": preflight[\\"status\\"], \\"csv_status\\": contracts[\\"status\\"],\\n               \\"discovery\\": {key: discovery[key] for key in (\\n                   \\"status\\", \\"scan_complete\\", \\"directories_checked\\", \\"entries_checked\\",\\n                   \\"limit_reached\\", \\"error_count\\", \\"skipped_link_count\\")}}\\n    if contracts[\\"status\\"] == \\"VALID\\":\\n        summary.update({\\n            \\"csv_sha256\\": dict(contracts[\\"sha256\\"]),\\n            \\"counts\\": {key: contracts[key][\\"row_count\\"] for key in (\\"train\\", \\"test\\", \\"train_series\\", \\"test_series\\")},\\n            \\"train_test_study_overlap_count\\": contracts[\\"train_test_study_overlap_count\\"],\\n            \\"train_test_series_overlap_count\\": contracts[\\"train_test_series_overlap_count\\"],\\n        })\\n    # No discovery paths, UID tables, arbitrary metadata categories or CSV error text.\\n    return summary\\n\\n\\ndef _payload(report, config):\\n    payload = json.dumps(report, sort_keys=True, indent=2, allow_nan=False) + \\"\\\\n\\"\\n    if len(payload.encode(\\"utf-8\\")) > config.max_report_bytes:\\n        raise InputError(\\"REPORT_SIZE_LIMIT_EXCEEDED\\")\\n    return payload\\n\\n\\ndef run_phase_c(config: PhaseCConfig, source_identity=None, *, preflight=None):\\n    if not isinstance(config, PhaseCConfig):\\n        raise TypeError(\\"run_phase_c requires validated PhaseCConfig\\")\\n    from .report_rules import freeze_rules, extract_report\\n    from .report_agreement import evaluate_candidates\\n\\n    started = time.perf_counter()\\n    source_identity = _validated_source_identity(source_identity)\\n    csv_config = config.preflight_config()\\n    # Freeze before extraction and any report-candidate agreement.\\n    frozen = freeze_rules()\\n    identity = frozen.identity()\\n    frozen.verify()\\n    if preflight is None:\\n        preflight = run_preflight(csv_config, source_identity=source_identity)\\n    elif (preflight.get(\\"configuration\\") != csv_config.to_dict()\\n          or preflight.get(\\"source_identity\\") != (source_identity or {\\"status\\": \\"NOT_SUPPLIED\\"})):\\n        raise InputError(\\"CSV preflight is stale; rerun with the current configuration and source\\")\\n    valid = preflight[\\"csv_contracts\\"].get(\\"status\\") == \\"VALID\\"\\n    checks = {name: {\\"status\\": \\"NOT_ASSESSED\\"} for name in (\\n        \\"report_quality\\", \\"rule_extraction\\", \\"observed_label_agreement\\", \\"validation_feasibility\\", \\"input_immutability\\")}\\n    checks[\\"csv_preflight\\"] = {\\"status\\": \\"PASS\\" if valid else \\"FAIL\\" if preflight[\\"csv_contracts\\"].get(\\"status\\") == \\"INVALID\\" else \\"BLOCKED\\"}\\n    checks[\\"rule_freeze\\"] = {\\"status\\": \\"PASS\\", \\"identity\\": identity,\\n                             \\"freeze_scope\\": \\"Immutable rule content frozen before extraction and agreement; no real-label tuning.\\",\\n                             \\"engine_source_sha256\\": (source_identity or {}).get(\\"source_sha256\\", {}).get(\\"report_rules.py\\")}\\n    if valid:\\n        try:\\n            root = safe_absolute(preflight[\\"discovery\\"][\\"selected_root\\"])\\n            tables, hashes = {}, {}\\n            for name in MANIFEST_NAMES:\\n                _, tables[name], hashes[name] = read_csv_manifest(root / name, csv_config)\\n            if hashes != preflight[\\"csv_contracts\\"][\\"sha256\\"]:\\n                raise InputError(\\"CSV_CHANGED_SINCE_PREFLIGHT\\")\\n            train = tables[\\"train.csv\\"]\\n            checks[\\"report_quality\\"] = aggregate_reports(train)\\n            extraction_started = time.perf_counter()\\n            candidates = []  # Private in-memory objects, separate from source targets.\\n            report_states = Counter()\\n            limit_reached = False\\n            for row in train:\\n                if time.perf_counter() - extraction_started >= config.max_extraction_seconds:\\n                    limit_reached = True\\n                    break\\n                candidate = extract_report(row[\\"Report\\"], frozen, max_characters=config.max_report_chars,\\n                                           max_clauses=config.max_report_clauses)\\n                candidates.append(candidate)\\n                report_states[candidate[\\"report_status\\"]] += 1\\n            frozen.verify()\\n            checks[\\"rule_extraction\\"] = {\\n                \\"status\\": \\"BLOCKED\\" if limit_reached else \\"PASS\\", \\"rule_identity\\": identity,\\n                \\"reports_processed\\": len(candidates), \\"reports_total\\": len(train),\\n                \\"report_status_counts\\": dict(sorted(report_states.items())),\\n                \\"time_limit_reached\\": limit_reached, \\"elapsed_seconds\\": time.perf_counter() - extraction_started,\\n                \\"limits\\": {\\"max_report_chars\\": config.max_report_chars, \\"max_report_clauses\\": config.max_report_clauses,\\n                           \\"max_extraction_seconds\\": config.max_extraction_seconds},\\n                \\"scope\\": \\"Frozen English rule candidates only; unsupported, ambiguous or bounded-out reports abstain. No per-study outputs retained.\\",\\n            }\\n            if not limit_reached:\\n                agreement = evaluate_candidates(train, candidates, identity)\\n                checks[\\"observed_label_agreement\\"] = {\\"status\\": \\"PASS\\", **agreement}\\n            del candidates\\n            checks[\\"validation_feasibility\\"] = assess_validation(train, tables[\\"train_series.csv\\"], preflight[\\"csv_contracts\\"][\\"train\\"])\\n            after = {name: read_csv_manifest(root / name, csv_config)[2] for name in MANIFEST_NAMES}\\n            checks[\\"input_immutability\\"] = {\\"status\\": \\"PASS\\" if after == hashes else \\"FAIL\\",\\n                                             \\"csv_sha256\\": after, \\"scope\\": \\"All five CSV hashes before/after; no DICOM or pixel reads\\"}\\n        except (InputError, ContractError, OSError) as error:\\n            checks[\\"input_or_contract\\"] = {\\"status\\": \\"BLOCKED\\", \\"error_category\\": type(error).__name__}\\n    if any(check[\\"status\\"] not in CHECK_STATES for check in checks.values()):\\n        raise ValueError(\\"Unrecognized Phase C check status\\")\\n    states = {check[\\"status\\"] for check in checks.values()}\\n    status = \\"FAIL\\" if \\"FAIL\\" in states else \\"BLOCKED\\" if {\\"BLOCKED\\", \\"NOT_ASSESSED\\"} & states else \\"PASS\\"\\n    report = {\\n        \\"schema\\": \\"rsna-knee-phase-c-v1\\", \\"phase\\": \\"PHASE_C\\", \\"status\\": status,\\n        \\"created_utc\\": datetime.now(timezone.utc).isoformat(), \\"configuration\\": config.to_dict(),\\n        \\"source_identity\\": source_identity or {\\"status\\": \\"NOT_SUPPLIED\\"},\\n        \\"rule_identity\\": identity, \\"preflight\\": _csv_summary(preflight), \\"checks\\": checks,\\n        \\"runtime\\": {\\"python\\": platform.python_version(), \\"platform\\": platform.system(), \\"gpu_used\\": False,\\n                    \\"dicom_pixels_decoded\\": 0, \\"elapsed_seconds\\": time.perf_counter() - started},\\n        \\"evaluation_status\\": \\"EXPLORATORY\\", \\"independent_validation\\": False,\\n        \\"training_ready\\": False, \\"split_approved\\": False, \\"model_metrics\\": None,\\n        \\"gold_resource\\": {\\"status\\": \\"NOT_ESTABLISHED\\", \\"independent_provenance_verified\\": False},\\n        \\"unassessed\\": [\\"real-rule reliability without Kaggle aggregate evidence\\", \\"dataset-wide trustworthy patient mapping\\",\\n                       \\"independent gold provenance\\", \\"locked twelve-target validation\\", \\"image-model performance\\"],\\n        \\"next_gate\\": \\"Review real aggregate evidence and validation design before training or rule changes informed by observed-label agreement.\\",\\n    }\\n    _payload(report, config)  # Size guard before notebook output, not just the write.\\n    return report\\n\\n\\ndef write_phase_c_report(report, config):\\n    if not isinstance(config, PhaseCConfig):\\n        raise TypeError(\\"write_phase_c_report requires validated PhaseCConfig\\")\\n    output, source = safe_absolute(config.output_dir), safe_absolute(config.input_base)\\n    if output.is_relative_to(source) or source.is_relative_to(output):\\n        raise InputError(\\"Phase C output must be separate from input_base\\")\\n    payload = _payload(report, config)\\n    output.mkdir(parents=True, exist_ok=True)\\n    safe_absolute(output)\\n    stamp = datetime.now(timezone.utc).strftime(\\"%Y%m%dT%H%M%S%fZ\\")\\n    path = output / f\\"phase_c_{stamp}_{uuid.uuid4().hex[:8]}.json\\"\\n    with path.open(\\"x\\", encoding=\\"utf-8\\", newline=\\"\\\\n\\") as stream:\\n        stream.write(payload)\\n    return path\\n","phase_c_config":"\\"\\"\\"CSV-only Phase C configuration; no DICOM, environment or network settings.\\"\\"\\"\\n\\nfrom dataclasses import dataclass, fields\\n\\nfrom .config import AuditConfig, ConfigError, _read_config_values\\n\\n\\n@dataclass(frozen=True)\\nclass PhaseCConfig(AuditConfig):\\n    inspect_csv: bool = True\\n    max_file_bytes: int = 32 * 1024 * 1024\\n    max_report_chars: int = 20000\\n    max_report_clauses: int = 256\\n    max_extraction_seconds: int = 120\\n\\n    def __post_init__(self):\\n        super().__post_init__()\\n        if not self.inspect_csv:\\n            raise ConfigError(\\"Phase C requires CSV preflight\\")\\n        if self.max_csv_rows > 100000:\\n            raise ConfigError(\\"Phase C permits at most 100000 CSV rows per manifest\\")\\n        for name, lower, upper in (\\n            (\\"max_report_chars\\", 1, 20000),\\n            (\\"max_report_clauses\\", 1, 256),\\n            (\\"max_extraction_seconds\\", 1, 600),\\n        ):\\n            value = getattr(self, name)\\n            if type(value) is not int or not lower <= value <= upper:\\n                raise ConfigError(f\\"{name} must be an integer in [{lower}, {upper}]\\")\\n\\n    def preflight_config(self) -> AuditConfig:\\n        return AuditConfig.from_dict({field.name: getattr(self, field.name) for field in fields(AuditConfig)})\\n\\n\\ndef load_phase_c_config(path) -> PhaseCConfig:\\n    return PhaseCConfig.from_dict(_read_config_values(path))\\n","phase_c_validation":"\\"\\"\\"Evidence needed for grouping and a prospective leakage-safe study design.\\"\\"\\"\\n\\nfrom collections import defaultdict\\n\\nfrom .contracts import STUDY_ID\\nfrom .validation_support import summarize_validation\\n\\n\\nPATIENT_COLUMNS = (\\"PatientID\\", \\"PatientId\\", \\"patient_id\\", \\"PatientInstanceUID\\")\\nFOLD_SAFEGUARDS = (\\n    \\"Keep all MRI series and slices of a study in one partition.\\",\\n    \\"Require proven dataset-wide patient mapping before claiming patient independence; keep repeated studies together when verified.\\",\\n    \\"Fit weak-label generation, learned preprocessing, transformations and calibration only in each fold\'s training partition.\\",\\n    \\"Exclude validation-study reports and their candidates from the image model\'s training labels, vocabulary fitting and rule tuning.\\",\\n    \\"Use observed validation targets and their original masks only; candidate labels never replace gold targets.\\",\\n    \\"Use observed supervision in preference to candidates; record source, mask, rule hash and any weak-label weight separately.\\",\\n    \\"Retain UNKNOWN as a false loss/evaluation mask; never fill missing labels with negative targets.\\",\\n    \\"Preserve exact target order, study identifiers and sample-submission schema; split at study/patient level, never by slices.\\",\\n    \\"Keep test studies and hidden targets out of model selection; the image-only inference model must not require reports.\\",\\n    \\"Treat exploratory rule agreement and model-selection scores as development evidence, never final performance estimates.\\",\\n    \\"Freeze rules before agreement measurement; a change after inspecting observed agreement requires a new version and a declared development evaluation.\\",\\n)\\n\\n\\ndef assess_validation(train_rows, series_rows, csv_train_summary):\\n    \\"\\"\\"Inspect possible CSV grouping fields without guessing their provenance.\\n\\n    Merely having a PatientID column or successful sampled DICOM headers cannot\\n    establish a trustworthy dataset-wide patient mapping. No IDs or assignments\\n    are returned and no DICOM metadata or pixels are read in this phase.\\n    \\"\\"\\"\\n    support = summarize_validation(train_rows, csv_train_summary)\\n    columns = {}\\n    for table_name, rows in ((\\"train.csv\\", train_rows), (\\"train_series.csv\\", series_rows)):\\n        available = {}\\n        for name in PATIENT_COLUMNS:\\n            if not any(name in row for row in rows):\\n                continue\\n            mappings = defaultdict(set)\\n            nonempty = 0\\n            for row in rows:\\n                value = row.get(name)\\n                if isinstance(value, str) and value.strip():\\n                    nonempty += 1\\n                    mappings[row[STUDY_ID]].add(value)\\n            available[name] = {\\n                \\"rows_with_nonempty_candidate_value\\": nonempty,\\n                \\"row_count\\": len(rows), \\"studies_with_candidate_value\\": len(mappings),\\n                \\"studies_with_inconsistent_candidate_values\\": sum(len(values) > 1 for values in mappings.values()),\\n            }\\n        columns[table_name] = available\\n    return {\\n        \\"status\\": support[\\"status\\"], \\"study_grouped_support\\": support,\\n        \\"patient_grouped_validation\\": {\\n            \\"status\\": \\"NOT_ASSESSED\\", \\"candidate_csv_fields\\": columns,\\n            \\"mapping_trustworthy\\": False,\\n            \\"reason\\": \\"No authoritative, complete, consistent dataset-wide patient map and provenance have been established. Candidate columns and sampled PatientID are insufficient.\\",\\n            \\"metadata_scope\\": \\"CSV field presence/consistency only; prior Phase B headers were sampled; no new DICOM reads.\\",\\n        },\\n        \\"prospective_design\\": {\\n            \\"status\\": \\"REVIEW_REQUIRED\\", \\"split_approved\\": False,\\n            \\"proposal\\": \\"Review a small two-fold study/patient-grouped exploratory design first; choose fold count only after measured per-class and joint support. Require an independent holdout or locked nested design for final estimates.\\",\\n            \\"patient_grouping_consequence\\": \\"Grouping repeated studies can reduce independent class support and make a marginally feasible study split impossible.\\",\\n            \\"label_sources\\": {\\n                \\"OBSERVED\\": \\"Original verified target and mask; primary reference supervision.\\",\\n                \\"REPORT_CANDIDATE\\": \\"Separate tri-state weak supervision from frozen rules on training-partition reports only; never gold truth.\\",\\n                \\"UNKNOWN\\": \\"No supervised target; false mask, no fabricated negative.\\",\\n            },\\n            \\"future_weak_labels\\": \\"Use training-only candidates on previously unknown cells with separately reviewed weights and quality gates; observed labels take precedence. Do not generate validation weak targets for training.\\",\\n            \\"safeguards\\": list(FOLD_SAFEGUARDS),\\n        },\\n        \\"warnings\\": [\\"Sparse labels may prevent a reliable twelve-target validation estimate.\\",\\n                     \\"A small class-coverage witness or successful audit does not approve a split.\\",\\n                     \\"No independent gold resource is established from the observed study count.\\"],\\n        \\"split_approved\\": False, \\"training_ready\\": False,\\n    }\\n","report_agreement":"\\"\\"\\"Aggregate exploratory agreement of frozen report candidates with masked truth.\\n\\nThese are candidate-rule agreement diagnostics, not image-model performance.\\nOnly verified observed binary cells participate in confusion counts. An\\nabstention remains UNKNOWN, including when an observed target is positive.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport math\\nimport re\\nfrom collections.abc import Mapping, Sequence\\nfrom typing import Any\\n\\nfrom .contracts import ContractError, TARGETS, label_vector\\n\\nRULE_VERSION = \\"rsna-report-rules-v1.0.0\\"\\nENGINE_POLICY = \\"conservative-english-clause-v1\\"\\nLOW_SUPPORT_THRESHOLD = 10\\nMAX_EVALUATION_ROWS = 100_000\\n_STATES = frozenset((\\"POSITIVE\\", \\"NEGATIVE\\", \\"UNKNOWN\\"))\\n_IDENTITY_KEYS = frozenset((\\"version\\", \\"sha256\\", \\"engine_policy\\"))\\n_Z95 = 1.959963984540054\\n\\n\\ndef wilson95(numerator: int, denominator: int) -> list[float] | None:\\n    \\"\\"\\"Nominal two-sided 95% Wilson score interval for a binomial proportion.\\n\\n    This interval does not establish independence, provenance, or clinical\\n    reliability. Undefined proportions receive no interval.\\n    \\"\\"\\"\\n    if (\\n        type(numerator) is not int or type(denominator) is not int\\n        or denominator < 0 or numerator < 0 or numerator > denominator\\n    ):\\n        raise ContractError(\\"agreement proportion requires valid integer counts\\")\\n    if denominator == 0:\\n        return None\\n    proportion = numerator / denominator\\n    z_squared = _Z95 * _Z95\\n    normalizer = 1.0 + z_squared / denominator\\n    center = (proportion + z_squared / (2.0 * denominator)) / normalizer\\n    radius = (\\n        _Z95 * math.sqrt(\\n            proportion * (1.0 - proportion) / denominator\\n            + z_squared / (4.0 * denominator * denominator)\\n        ) / normalizer\\n    )\\n    return [max(0.0, center - radius), min(1.0, center + radius)]\\n\\n\\ndef _proportion(numerator: int, denominator: int) -> dict[str, Any]:\\n    return {\\n        \\"value\\": numerator / denominator if denominator else None,\\n        \\"numerator\\": numerator,\\n        \\"denominator\\": denominator,\\n        \\"wilson95\\": wilson95(numerator, denominator),\\n    }\\n\\n\\ndef _identity(value: Any) -> dict[str, str]:\\n    if (\\n        not isinstance(value, Mapping) or set(value) != _IDENTITY_KEYS\\n        or value.get(\\"version\\") != RULE_VERSION\\n        or value.get(\\"engine_policy\\") != ENGINE_POLICY\\n        or not isinstance(value.get(\\"sha256\\"), str)\\n        or re.fullmatch(r\\"[0-9a-f]{64}\\", value[\\"sha256\\"]) is None\\n    ):\\n        raise ContractError(\\"agreement requires a valid frozen rule identity\\")\\n    return {\\"version\\": RULE_VERSION, \\"sha256\\": value[\\"sha256\\"], \\"engine_policy\\": ENGINE_POLICY}\\n\\n\\ndef _validate_candidate(value: Any, identity: dict[str, str]) -> None:\\n    if not isinstance(value, Mapping) or _identity(value.get(\\"rule_identity\\")) != identity:\\n        raise ContractError(\\"agreement candidate rule identity mismatch\\")\\n    targets = value.get(\\"targets\\")\\n    if not isinstance(targets, Mapping) or set(targets) != set(TARGETS):\\n        raise ContractError(\\"agreement candidate target contract is invalid\\")\\n    for target in TARGETS:\\n        candidate = targets[target]\\n        if (\\n            not isinstance(candidate, Mapping)\\n            or not isinstance(candidate.get(\\"state\\"), str)\\n            or candidate[\\"state\\"] not in _STATES\\n            or type(candidate.get(\\"conflict\\")) is not bool\\n            or (candidate[\\"conflict\\"] and candidate[\\"state\\"] != \\"UNKNOWN\\")\\n        ):\\n            raise ContractError(\\"agreement candidate state contract is invalid\\")\\n\\n\\ndef _warnings(support: dict[str, int], metrics: dict[str, Any]) -> list[str]:\\n    warnings = []\\n    if support[\\"observed\\"] == 0:\\n        warnings.append(\\"NO_OBSERVED_TRUTH: agreement is not assessed for this target.\\")\\n    if support[\\"positive\\"] < LOW_SUPPORT_THRESHOLD:\\n        warnings.append(\\"LOW_POSITIVE_SUPPORT: fewer than 10 observed positive cells.\\")\\n    if support[\\"negative\\"] < LOW_SUPPORT_THRESHOLD:\\n        warnings.append(\\"LOW_NEGATIVE_SUPPORT: fewer than 10 observed negative cells.\\")\\n    for name in (\\n        \\"agreement\\", \\"positive_predictive_value\\", \\"selective_sensitivity\\",\\n        \\"selective_specificity\\", \\"negative_predictive_value\\", \\"detection_sensitivity\\",\\n    ):\\n        if metrics[name][\\"denominator\\"] < LOW_SUPPORT_THRESHOLD:\\n            warnings.append(f\\"LOW_METRIC_DENOMINATOR: {name} uses fewer than 10 cells.\\")\\n    return warnings\\n\\n\\ndef evaluate_candidates(\\n    train_rows: Sequence[Mapping[str, Any]],\\n    candidates: Sequence[Mapping[str, Any]],\\n    rule_identity: Mapping[str, str],\\n) -> dict[str, Any]:\\n    \\"\\"\\"Compare aligned private candidates to observed labels and emit counts only.\\n\\n    The caller freezes the rule set before extraction and evaluation. Every\\n    candidate must carry the identical frozen identity. Alignment is by input\\n    order, not by copied study identifiers. Reports, identifiers, reason codes,\\n    arbitrary candidate fields, and per-study predictions never enter output.\\n    \\"\\"\\"\\n    identity = _identity(rule_identity)\\n    if (\\n        not isinstance(train_rows, Sequence) or isinstance(train_rows, (str, bytes))\\n        or not isinstance(candidates, Sequence) or isinstance(candidates, (str, bytes))\\n        or len(train_rows) != len(candidates)\\n        or len(train_rows) > MAX_EVALUATION_ROWS\\n    ):\\n        raise ContractError(\\"agreement requires aligned bounded row and candidate sequences\\")\\n    # Check identity and candidate contracts before computing any agreement.\\n    for candidate in candidates:\\n        _validate_candidate(candidate, identity)\\n    counts = {\\n        target: {\\n            \\"candidate_counts\\": {\\"positive\\": 0, \\"negative\\": 0, \\"unknown\\": 0, \\"conflict\\": 0},\\n            \\"truth_support\\": {\\"observed\\": 0, \\"positive\\": 0, \\"negative\\": 0, \\"unknown\\": 0},\\n            \\"abstained_unknown_by_truth\\": {\\"positive\\": 0, \\"negative\\": 0, \\"unknown\\": 0},\\n            \\"classified_truth_counts\\": {\\"positive\\": 0, \\"negative\\": 0, \\"unknown\\": 0},\\n            \\"confusion\\": {\\"TP\\": 0, \\"FP\\": 0, \\"TN\\": 0, \\"FN\\": 0},\\n        }\\n        for target in TARGETS\\n    }\\n    observed_studies = 0\\n    for row, candidate in zip(train_rows, candidates):\\n        try:\\n            labels = label_vector(row)\\n        except (ContractError, TypeError) as error:\\n            raise ContractError(\\"agreement source label contract is invalid\\") from error\\n        if \\"invalid\\" in labels[\\"states\\"]:\\n            raise ContractError(\\"agreement source contains an invalid label cell\\")\\n        observed_studies += any(labels[\\"observed_mask\\"])\\n        for index, target in enumerate(TARGETS):\\n            target_counts = counts[target]\\n            target_candidate = candidate[\\"targets\\"][target]\\n            state = target_candidate[\\"state\\"]\\n            target_counts[\\"candidate_counts\\"][state.lower()] += 1\\n            target_counts[\\"candidate_counts\\"][\\"conflict\\"] += target_candidate[\\"conflict\\"]\\n            observed = labels[\\"observed_mask\\"][index]\\n            truth_class = (\\n                \\"positive\\" if labels[\\"values\\"][index] == 1.0 else \\"negative\\"\\n            ) if observed else \\"unknown\\"\\n            target_counts[\\"truth_support\\"][truth_class] += 1\\n            target_counts[\\"truth_support\\"][\\"observed\\"] += observed\\n            if state == \\"UNKNOWN\\":\\n                target_counts[\\"abstained_unknown_by_truth\\"][truth_class] += 1\\n                continue\\n            target_counts[\\"classified_truth_counts\\"][truth_class] += 1\\n            if not observed:\\n                continue\\n            confusion_key = (\\n                \\"TP\\" if truth_class == \\"positive\\" else \\"FP\\"\\n            ) if state == \\"POSITIVE\\" else (\\n                \\"FN\\" if truth_class == \\"positive\\" else \\"TN\\"\\n            )\\n            target_counts[\\"confusion\\"][confusion_key] += 1\\n    for target_counts in counts.values():\\n        candidate_counts = target_counts[\\"candidate_counts\\"]\\n        support = target_counts[\\"truth_support\\"]\\n        confusion = target_counts[\\"confusion\\"]\\n        tp, fp, tn, fn = (confusion[key] for key in (\\"TP\\", \\"FP\\", \\"TN\\", \\"FN\\"))\\n        classified_observed = tp + fp + tn + fn\\n        metrics = {\\n            \\"agreement\\": _proportion(tp + tn, classified_observed),\\n            \\"positive_predictive_value\\": _proportion(tp, tp + fp),\\n            \\"selective_sensitivity\\": _proportion(tp, tp + fn),\\n            \\"selective_specificity\\": _proportion(tn, tn + fp),\\n            \\"negative_predictive_value\\": _proportion(tn, tn + fn),\\n            \\"detection_sensitivity\\": _proportion(tp, support[\\"positive\\"]),\\n        }\\n        target_counts.update({\\n            \\"coverage\\": _proportion(candidate_counts[\\"positive\\"] + candidate_counts[\\"negative\\"], len(train_rows)),\\n            \\"observed_truth_coverage\\": _proportion(classified_observed, support[\\"observed\\"]),\\n            \\"metrics\\": metrics,\\n            \\"warnings\\": _warnings(support, metrics),\\n        })\\n    return {\\n        \\"status\\": \\"PASS\\",\\n        \\"evaluation_status\\": \\"EXPLORATORY\\",\\n        \\"independent_validation\\": False,\\n        \\"rule_identity\\": identity,\\n        \\"rule_identity_consistency_verified\\": True,\\n        \\"row_count\\": len(train_rows),\\n        \\"observed_label_study_count\\": observed_studies,\\n        \\"observed_label_cell_count\\": sum(count[\\"truth_support\\"][\\"observed\\"] for count in counts.values()),\\n        \\"per_target\\": counts,\\n        \\"warning_policy\\": {\\"low_support_threshold\\": LOW_SUPPORT_THRESHOLD, \\"comparison\\": \\"strictly_below\\"},\\n        \\"metric_definitions\\": {\\n            \\"agreement\\": \\"(TP + TN) / classified observed truth cells; unknown candidates excluded.\\",\\n            \\"positive_predictive_value\\": \\"TP / (TP + FP), observed truth only.\\",\\n            \\"selective_sensitivity\\": \\"TP / (TP + FN), classified observed positives only.\\",\\n            \\"selective_specificity\\": \\"TN / (TN + FP), classified observed negatives only.\\",\\n            \\"negative_predictive_value\\": \\"TN / (TN + FN), observed truth only.\\",\\n            \\"detection_sensitivity\\": \\"TP / all observed positives; abstentions are not detections and are never assigned negative labels.\\",\\n        },\\n        \\"limitations\\": [\\n            \\"Candidate-rule agreement is not image-model performance or competition accuracy.\\",\\n            \\"Rules must be frozen before evaluation; the same observed labels must not be used to tune rules and claim independent validation.\\",\\n            \\"Small observed-label samples, including 58 studies, cannot establish reliable twelve-target validation by their count alone.\\",\\n            \\"Wilson intervals are nominal binomial intervals; patient independence and representative sampling remain unverified.\\",\\n            \\"Unknown truth cells are excluded from every confusion count and agreement metric.\\",\\n        ],\\n        \\"model_metrics\\": None,\\n    }\\n","report_quality":"\\"\\"\\"Aggregate report-quality indicators without exporting report content.\\r\\n\\r\\nThe script categories below are character heuristics, not language detection.\\r\\nInput rows and target labels are never changed or converted into weak labels.\\r\\n\\"\\"\\"\\r\\n\\r\\nfrom __future__ import annotations\\r\\n\\r\\nfrom collections import Counter\\r\\nfrom collections.abc import Mapping\\r\\nimport math\\r\\nimport statistics\\r\\nimport unicodedata\\r\\nfrom typing import Any\\r\\n\\r\\n\\r\\nSHORT_REPORT_CHARACTERS = 20\\r\\nLONG_REPORT_CHARACTERS = 10_000\\r\\nSCRIPT_CATEGORIES = (\\r\\n    \\"latin_only\\", \\"devanagari_only\\", \\"latin_and_devanagari\\",\\r\\n    \\"latin_and_other_letters\\", \\"devanagari_and_other_letters\\",\\r\\n    \\"latin_devanagari_and_other_letters\\", \\"other_letters_only\\", \\"no_letters\\",\\r\\n)\\r\\n\\r\\n\\r\\ndef _percentile(values: list[int], fraction: float) -> float | None:\\r\\n    \\"\\"\\"Linearly interpolate sorted values at (n - 1) * fraction.\\"\\"\\"\\r\\n    if not values:\\r\\n        return None\\r\\n    position = (len(values) - 1) * fraction\\r\\n    lower, upper = math.floor(position), math.ceil(position)\\r\\n    return values[lower] + (values[upper] - values[lower]) * (position - lower)\\r\\n\\r\\n\\r\\ndef _distribution(values: list[int]) -> dict[str, int | float | None]:\\r\\n    ordered = sorted(values)\\r\\n    return {\\r\\n        \\"count\\": len(ordered), \\"min\\": ordered[0] if ordered else None,\\r\\n        \\"median\\": statistics.median(ordered) if ordered else None,\\r\\n        \\"p90\\": _percentile(ordered, 0.90), \\"p99\\": _percentile(ordered, 0.99),\\r\\n        \\"max\\": ordered[-1] if ordered else None,\\r\\n    }\\r\\n\\r\\n\\r\\ndef _script_category(latin: bool, devanagari: bool, other: bool) -> str:\\r\\n    return {\\r\\n        (True, False, False): \\"latin_only\\",\\r\\n        (False, True, False): \\"devanagari_only\\",\\r\\n        (True, True, False): \\"latin_and_devanagari\\",\\r\\n        (True, False, True): \\"latin_and_other_letters\\",\\r\\n        (False, True, True): \\"devanagari_and_other_letters\\",\\r\\n        (True, True, True): \\"latin_devanagari_and_other_letters\\",\\r\\n        (False, False, True): \\"other_letters_only\\",\\r\\n        (False, False, False): \\"no_letters\\",\\r\\n    }[(latin, devanagari, other)]\\r\\n\\r\\n\\r\\ndef aggregate_reports(train_rows: list[dict[str, Any]]) -> dict[str, Any]:\\r\\n    \\"\\"\\"Inspect all supplied Report cells; return fixed categories and counts.\\r\\n\\r\\n    The caller bounds manifest size before parsing. Duplicate normalization is\\r\\n    NFKC, case folding and whitespace collapse, used in memory only. Empty\\r\\n    reports do not contribute to duplicate or length distributions.\\r\\n    \\"\\"\\"\\r\\n    normalized: Counter[str] = Counter()\\r\\n    scripts = {name: 0 for name in SCRIPT_CATEGORIES}\\r\\n    flags = {name: 0 for name in (\\r\\n        \\"unexpected_control_characters\\", \\"format_characters\\", \\"replacement_characters\\",\\r\\n        \\"private_use_characters\\", \\"surrogate_characters\\", \\"unassigned_characters\\",\\r\\n        \\"other_symbols\\", \\"unusual_characters\\", \\"very_short_reports\\", \\"very_long_reports\\",\\r\\n    )}\\r\\n    lengths: list[int] = []\\r\\n    words: list[int] = []\\r\\n    missing = empty = invalid = latin_reports = devanagari_reports = 0\\r\\n    latin_letters = devanagari_letters = other_letters = 0\\r\\n    latin_characters = devanagari_characters = 0\\r\\n    for row in train_rows:\\r\\n        if not isinstance(row, Mapping):\\r\\n            invalid += 1\\r\\n            continue\\r\\n        report = row.get(\\"Report\\")\\r\\n        if report is None:\\r\\n            missing += 1\\r\\n            continue\\r\\n        if not isinstance(report, str):\\r\\n            invalid += 1\\r\\n            continue\\r\\n        if not report.strip():\\r\\n            empty += 1\\r\\n            continue\\r\\n        lengths.append(len(report))\\r\\n        words.append(len(report.split()))\\r\\n        normalized[\\" \\".join(unicodedata.normalize(\\"NFKC\\", report).casefold().split())] += 1\\r\\n        encountered: set[str] = set()\\r\\n        latin = devanagari = other = False\\r\\n        for character in report:\\r\\n            category = unicodedata.category(character)\\r\\n            name = unicodedata.name(character, \\"\\")\\r\\n            if \\"LATIN\\" in name:\\r\\n                latin = True\\r\\n                latin_characters += 1\\r\\n            elif \\"DEVANAGARI\\" in name:\\r\\n                devanagari = True\\r\\n                devanagari_characters += 1\\r\\n            if category.startswith(\\"L\\"):\\r\\n                if \\"LATIN\\" in name:\\r\\n                    latin_letters += 1\\r\\n                elif \\"DEVANAGARI\\" in name:\\r\\n                    devanagari_letters += 1\\r\\n                else:\\r\\n                    other = True\\r\\n                    other_letters += 1\\r\\n            if category == \\"Cc\\" and character not in \\"\\\\t\\\\r\\\\n\\":\\r\\n                encountered.add(\\"unexpected_control_characters\\")\\r\\n            if category == \\"Cf\\":\\r\\n                encountered.add(\\"format_characters\\")\\r\\n            if character == \\"\\\\ufffd\\":\\r\\n                encountered.add(\\"replacement_characters\\")\\r\\n            for expected, flag in ((\\"Co\\", \\"private_use_characters\\"), (\\"Cs\\", \\"surrogate_characters\\"),\\r\\n                                   (\\"Cn\\", \\"unassigned_characters\\"), (\\"So\\", \\"other_symbols\\")):\\r\\n                if category == expected:\\r\\n                    encountered.add(flag)\\r\\n        for flag in encountered:\\r\\n            flags[flag] += 1\\r\\n        flags[\\"unusual_characters\\"] += bool(encountered)\\r\\n        flags[\\"very_short_reports\\"] += len(report.strip()) < SHORT_REPORT_CHARACTERS\\r\\n        flags[\\"very_long_reports\\"] += len(report) > LONG_REPORT_CHARACTERS\\r\\n        latin_reports += latin\\r\\n        devanagari_reports += devanagari\\r\\n        scripts[_script_category(latin, devanagari, other)] += 1\\r\\n    duplicate_groups = sum(count > 1 for count in normalized.values())\\r\\n    duplicates = sum(count - 1 for count in normalized.values())\\r\\n    return {\\r\\n        \\"status\\": \\"FAIL\\" if invalid else \\"PASS\\", \\"scope\\": \\"All supplied training Report cells\\",\\r\\n        \\"row_count\\": len(train_rows), \\"nonempty_report_count\\": len(lengths),\\r\\n        \\"missing_report_count\\": missing, \\"empty_report_count\\": empty,\\r\\n        \\"empty_or_missing_report_count\\": missing + empty, \\"invalid_report_type_count\\": invalid,\\r\\n        \\"normalized_duplicate_report_count\\": duplicates, \\"normalized_duplicate_group_count\\": duplicate_groups,\\r\\n        \\"reports_in_duplicate_groups\\": sum(count for count in normalized.values() if count > 1),\\r\\n        \\"normalization\\": \\"NFKC, casefold, whitespace collapse; nonempty reports only\\",\\r\\n        \\"character_count\\": _distribution(lengths), \\"whitespace_word_count\\": _distribution(words),\\r\\n        \\"percentile_method\\": \\"Linear interpolation at (n - 1) * quantile\\",\\r\\n        \\"unicode_script_indicators\\": {\\r\\n            \\"latin_present_report_count\\": latin_reports,\\r\\n            \\"devanagari_present_report_count\\": devanagari_reports,\\r\\n            \\"latin_character_count\\": latin_characters,\\r\\n            \\"devanagari_character_count\\": devanagari_characters,\\r\\n            \\"latin_letter_count\\": latin_letters, \\"devanagari_letter_count\\": devanagari_letters,\\r\\n            \\"other_letter_count\\": other_letters, \\"heuristic_report_categories\\": scripts,\\r\\n        },\\r\\n        \\"quality_flag_report_counts\\": flags,\\r\\n        \\"flag_thresholds\\": {\\"very_short_trimmed_characters_less_than\\": SHORT_REPORT_CHARACTERS,\\r\\n                            \\"very_long_original_characters_greater_than\\": LONG_REPORT_CHARACTERS},\\r\\n        \\"quality_review_required\\": bool(missing or empty or invalid or duplicates or any(flags.values())),\\r\\n        \\"limitations\\": [\\r\\n            \\"Unicode character names indicate scripts, not reliable language identification.\\",\\r\\n            \\"Latin does not establish English; Devanagari does not establish Hindi.\\",\\r\\n            \\"Whitespace tokens are not linguistic words and may undercount scripts without word spaces.\\",\\r\\n            \\"Format characters and symbols can be legitimate; quality flags require human review.\\",\\r\\n            \\"Normalized duplicates do not establish patient identity or report correctness.\\",\\r\\n            \\"No report text, normalized strings, report-derived labels or identifiers are exported.\\",\\r\\n        ],\\r\\n        \\"weak_labels_created\\": False, \\"target_labels_changed\\": False,\\r\\n    }\\r\\n","report_rules":"\\"\\"\\"Frozen, conservative English report candidates; never gold target labels.\\n\\nThis small prototype recognizes explicit English findings in bounded clauses.\\nScript and language cues are rejection heuristics, not language identification.\\nReport content is used in memory only and is never included in returned values.\\nThe canonical rule hash covers definitions and parsing policies; the notebook\\nseparately verifies this module\'s source hash to identify the actual engine.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom dataclasses import dataclass\\nimport hashlib\\nimport json\\nimport re\\nimport unicodedata\\nfrom typing import Any\\n\\nfrom .contracts import TARGETS\\n\\n\\nRULE_VERSION = \\"rsna-report-rules-v1.0.0\\"\\nENGINE_POLICY = \\"conservative-english-clause-v1\\"\\nSTATES = (\\"POSITIVE\\", \\"NEGATIVE\\", \\"UNKNOWN\\")\\nMAX_REPORT_CHARACTERS = 20_000\\nMAX_REPORT_CLAUSES = 256\\n\\n\\n@dataclass(frozen=True)\\nclass TargetRule:\\n    \\"\\"\\"A readable target definition with no mutable compiled-regex cache.\\"\\"\\"\\n\\n    target: str\\n    category: str\\n    mention: str\\n    abnormality: str\\n    normal: str\\n\\n\\ndef _definitions() -> tuple[TargetRule, ...]:\\n    ligament = r\\"\\\\b(?:tear(?:s|ing)?|torn|ruptur(?:e|ed)|disruption|discontinuity|avulsion|mucoid degeneration)\\\\b\\"\\n    meniscus = r\\"\\\\b(?:tear(?:s|ing)?|torn|ruptur(?:e|ed)|maceration|extrusion|degeneration|displaced fragment)\\\\b\\"\\n    normal = r\\"\\\\b(?:intact|normal|unremarkable|preserved)\\\\b\\"\\n    oa = r\\"\\\\b(?:oa|osteoarthritis|osteoarthrosis|degenerative joint disease|degenerative changes)\\\\b\\"\\n    return (\\n        TargetRule(\\"ACL\\", \\"structure\\", r\\"\\\\b(?:acl|anterior cruciate ligament)\\\\b\\", ligament, normal),\\n        TargetRule(\\"MCL\\", \\"structure\\", r\\"\\\\b(?:mcl|medial collateral ligament)\\\\b\\", ligament + r\\"|\\\\bsprain(?:ed)?\\\\b\\", normal),\\n        TargetRule(\\"Medial Meniscus\\", \\"structure\\", r\\"\\\\bmedial menisc(?:us|al)\\\\b\\", meniscus, normal),\\n        TargetRule(\\"Lateral Meniscus\\", \\"structure\\", r\\"\\\\blateral menisc(?:us|al)\\\\b\\", meniscus, normal),\\n        TargetRule(\\"Medial OA\\", \\"oa\\", r\\"\\\\bmedial(?:\\\\s+tibiofemoral)?\\\\s+compartment\\\\b|\\\\bmedial(?=\\\\s+(?:oa\\\\b|osteoarthritis\\\\b|osteoarthrosis\\\\b))\\", oa, \\"\\"),\\n        TargetRule(\\"Lateral OA\\", \\"oa\\", r\\"\\\\blateral(?:\\\\s+tibiofemoral)?\\\\s+compartment\\\\b|\\\\blateral(?=\\\\s+(?:oa\\\\b|osteoarthritis\\\\b|osteoarthrosis\\\\b))\\", oa, \\"\\"),\\n        TargetRule(\\"PF OA\\", \\"oa\\", r\\"\\\\b(?:patellofemoral|pf)(?:\\\\s+compartment)?\\\\b\\", oa, \\"\\"),\\n        TargetRule(\\"Effusion\\", \\"condition\\", r\\"\\\\b(?:joint\\\\s+)?effusions?\\\\b\\", \\"\\", \\"\\"),\\n        TargetRule(\\"Synovitis\\", \\"condition\\", r\\"\\\\bsynovitis\\\\b\\", \\"\\", \\"\\"),\\n        TargetRule(\\"Baker\'s\\", \\"condition\\", r\\"\\\\b(?:baker(?:[\'\\\\u2019]s|s)?\\\\s+cyst|popliteal cyst)\\\\b\\", \\"\\", \\"\\"),\\n        TargetRule(\\"Contusion\\", \\"condition\\", r\\"\\\\b(?:(?:bone|osseous|marrow)\\\\s+)?contusions?\\\\b|\\\\bbone bruis(?:e|ing)\\\\b\\", \\"\\", \\"\\"),\\n        TargetRule(\\"Fracture\\", \\"condition\\", r\\"\\\\bfractures?\\\\b\\", \\"\\", \\"\\"),\\n    )\\n\\n\\ndef _cues() -> tuple[tuple[str, str], ...]:\\n    return (\\n        (\\"clause_separator\\", r\\"[.!?;\\\\n\\\\r,]+|\\\\b(?:but|however|nevertheless)\\\\b\\"),\\n        (\\"coordinate\\", r\\"\\\\b(?:and|or|with|without)\\\\b\\"),\\n        (\\"negative\\", r\\"\\\\b(?:no|not|without|absent|absence of|negative for|not seen|not present|not identified|not demonstrated|not visualized|not detected|not evident|not appreciated|not torn|not ruptured|excluded)\\\\b\\"),\\n        (\\"immediate_negative\\", r\\"\\\\b(?:no|without|absence of|negative for)\\\\s+(?:(?:evidence|signs?|of|in|a|an|the|any|acute|current|complete|partial|tear|tears|rupture|sprain|injury|degenerative|changes|joint|disease|oa|osteoarthritis|osteoarthrosis)\\\\s+)*$\\"),\\n        (\\"post_negative\\", r\\"^\\\\s*(?:(?:is|are|was|were|remains?|tear|tears|rupture|sprain|injury|oa|osteoarthritis|osteoarthrosis|degenerative|changes|has|demonstrates?)\\\\s+)*(?:absent|not seen|not present|not identified|not demonstrated|not visualized|not detected|not evident|not appreciated|not torn|not ruptured|excluded|no\\\\s+(?:tear|injury|rupture|sprain|abnormality))\\\\b\\"),\\n        (\\"uncertain\\", r\\"\\\\b(?:possible|possibly|possibility|probable|probably|likely|unlikely|suspect(?:ed)?|suspicious|suspicion|suggest(?:s|ive|ion)?|equivocal|questionable|may|might|could|versus|vs|differential|rule out|r/o|cannot exclude|can\'t exclude|not excluded|cannot be excluded|cannot be ruled out|not ruled out|concern(?:ing)? for|risk of|at risk)\\\\b\\"),\\n        (\\"historical\\", r\\"\\\\b(?:history of|historical|previous|previously|prior|remote|past|old healed|healed|after healing|resolved|status post|postoperative|post-operative|repaired)\\\\b\\"),\\n        (\\"mention_only\\", r\\"\\\\b(?:evaluate for|evaluation for|assess for|assessment for|query|check for|screen for|clinical concern|clinical suspicion|evaluated|mentioned|discussed)\\\\b\\"),\\n        (\\"assertion\\", r\\"\\\\b(?:there is|there are|demonstrates?|reveals?|shows?|identified|observed|visualized|seen|present|noted|evident)\\\\b\\"),\\n        (\\"copula\\", r\\"\\\\b(?:is|are)\\\\b\\"),\\n        (\\"positive_qualifier\\", r\\"\\\\b(?:small|moderate|large|mild|marked|severe|trace|significant|acute|subchondral|nondisplaced|displaced|comminuted|impacted)\\\\b\\"),\\n        (\\"generic_abnormality\\", r\\"\\\\b(?:abnormality|abnormalities|injury|injuries|tear|tears|rupture|sprain|degeneration)\\\\b\\"),\\n        (\\"ignored_header\\", r\\"^(?:clinical history|clinical information|clinical indication|indication|history|technique|differential diagnosis|differential)\\\\s*:\\\\s*\\"),\\n        (\\"active_header\\", r\\"^(?:findings|impression|conclusion|results)\\\\s*:\\\\s*\\"),\\n        (\\"other_header\\", r\\"^([a-z][a-z ]{0,40})\\\\s*:\\\\s*\\"),\\n        (\\"header_boundary\\", r\\"\\\\b(?:clinical history|clinical information|clinical indication|indication|history|technique|differential diagnosis|differential|findings|impression|conclusion|results)\\\\s*:\\"),\\n        (\\"negated_normal\\", r\\"\\\\b(?:not|no)\\\\s+(?:intact|normal|unremarkable|preserved)\\\\b\\"),\\n        (\\"other_structure\\", r\\"\\\\b(?:pcl|posterior cruciate ligament|lcl|lateral collateral ligament|quadriceps|patellar tendon|skin|soft tissue)\\\\b\\"),\\n        (\\"bone_contusion\\", r\\"\\\\b(?:bone|osseous|marrow)\\\\s+contusions?\\\\b|\\\\bbone bruis(?:e|ing)\\\\b\\"),\\n        (\\"nonenglish_cue\\", r\\"\\\\b(?:hai|hain|nahi|nahin|tha|thi|mein|aur|dard|bina)\\\\b\\"),\\n    )\\n\\n\\n@dataclass(frozen=True)\\nclass FrozenRuleSet:\\n    \\"\\"\\"Immutable definitions captured before agreement evaluation.\\n\\n    verify() also detects deliberate object.__setattr__ tampering. It raises a\\n    fixed message and never prints the altered content.\\n    \\"\\"\\"\\n\\n    version: str\\n    engine_policy: str\\n    rules: tuple[TargetRule, ...]\\n    cues: tuple[tuple[str, str], ...]\\n    normalization_policy: tuple[str, ...]\\n    scope_policy: tuple[str, ...]\\n    sha256: str\\n\\n    def _canonical(self) -> bytes:\\n        payload = {\\n            \\"version\\": self.version, \\"engine_policy\\": self.engine_policy,\\n            \\"rules\\": [\\n                {\\"target\\": rule.target, \\"category\\": rule.category,\\n                 \\"mention\\": rule.mention, \\"abnormality\\": rule.abnormality,\\n                 \\"normal\\": rule.normal}\\n                for rule in self.rules\\n            ],\\n            \\"cues\\": [list(cue) for cue in self.cues],\\n            \\"normalization_policy\\": list(self.normalization_policy),\\n            \\"scope_policy\\": list(self.scope_policy),\\n        }\\n        return json.dumps(payload, sort_keys=True, separators=(\\",\\", \\":\\"), ensure_ascii=True).encode(\\"ascii\\")\\n\\n    def verify(self) -> None:\\n        \\"\\"\\"Reject mutation and malformed definitions before any extraction.\\"\\"\\"\\n        try:\\n            valid = (\\n                type(self.rules) is tuple and all(type(rule) is TargetRule for rule in self.rules)\\n                and all(all(type(value) is str for value in (rule.target, rule.category, rule.mention, rule.abnormality, rule.normal)) for rule in self.rules)\\n                and tuple(rule.target for rule in self.rules) == TARGETS\\n                and type(self.cues) is tuple and all(type(cue) is tuple and len(cue) == 2 and all(type(value) is str for value in cue) for cue in self.cues)\\n                and type(self.normalization_policy) is tuple and type(self.scope_policy) is tuple\\n                and all(type(value) is str for value in (*self.normalization_policy, *self.scope_policy))\\n                and self.version == RULE_VERSION and self.engine_policy == ENGINE_POLICY\\n                and hashlib.sha256(self._canonical()).hexdigest() == self.sha256\\n            )\\n        except (TypeError, ValueError, AttributeError):\\n            valid = False\\n        if not valid:\\n            raise ValueError(\\"frozen rule-set integrity check failed\\")\\n\\n    def identity(self) -> dict[str, str]:\\n        self.verify()\\n        return {\\"version\\": self.version, \\"sha256\\": self.sha256, \\"engine_policy\\": self.engine_policy}\\n\\n\\ndef freeze_rules() -> FrozenRuleSet:\\n    \\"\\"\\"Return a fresh immutable instance; no observed labels enter this API.\\"\\"\\"\\n    frozen = FrozenRuleSet(\\n        RULE_VERSION, ENGINE_POLICY, _definitions(), _cues(),\\n        (\\"Unicode NFKC\\", \\"casefold\\", \\"per-line whitespace collapse\\", \\"question mark adds uncertainty before clause splitting\\", \\"reject unexpected controls and non-Latin letters\\",\\n         \\"Latin script does not establish English; fixed rejection cues only\\"),\\n        (\\"bounded punctuation clauses\\", \\"history and indication sections ignored until an explicit findings section\\",\\n         \\"nearest structure with immediate coordinated negation; with and new copulas end inheritance\\", \\"comma after negation adds ambiguity\\", \\"explicit abnormality or absence required\\",\\n         \\"uncertainty and contradictions remain UNKNOWN\\", \\"no absent-term negative inference\\",\\n         \\"maximum source characters 20000\\", \\"maximum clauses 256\\", \\"maximum target context 160 characters\\"),\\n        \\"\\",\\n    )\\n    return FrozenRuleSet(\\n        frozen.version, frozen.engine_policy, frozen.rules, frozen.cues,\\n        frozen.normalization_policy, frozen.scope_policy,\\n        hashlib.sha256(frozen._canonical()).hexdigest(),\\n    )\\n\\n\\ndef _unknown_result(frozen: FrozenRuleSet, status: str, reason: str) -> dict[str, Any]:\\n    return {\\n        \\"targets\\": {target: {\\"state\\": \\"UNKNOWN\\", \\"conflict\\": False, \\"reason_codes\\": (reason,)} for target in TARGETS},\\n        \\"report_status\\": status, \\"rule_identity\\": frozen.identity(),\\n    }\\n\\n\\ndef _unsupported_characters(report: str) -> str | None:\\n    for char in report:\\n        category = unicodedata.category(char)\\n        if (category == \\"Cc\\" and char not in \\"\\\\t\\\\r\\\\n\\") or category in {\\"Cf\\", \\"Cs\\", \\"Co\\", \\"Cn\\"} or char == \\"\\\\ufffd\\":\\n            return \\"MALFORMED_UNICODE\\"\\n        if category.startswith(\\"L\\") and \\"LATIN\\" not in unicodedata.name(char, \\"\\"):\\n            return \\"UNSUPPORTED_SCRIPT\\"\\n    return None\\n\\n\\ndef _clauses(text: str, cues: dict[str, str]) -> list[tuple[str, bool]]:\\n    clauses: list[tuple[str, bool]] = []\\n    ignored_section = False\\n    text = re.sub(cues[\\"header_boundary\\"], lambda match: \\"\\\\n\\" + match.group(), text)\\n    for line in text.splitlines() or [text]:\\n        line = \\" \\".join(line.split())\\n        line = line.replace(\\"?\\", \\" possible;\\")\\n        ignored = re.match(cues[\\"ignored_header\\"], line)\\n        active = re.match(cues[\\"active_header\\"], line)\\n        other = re.match(cues[\\"other_header\\"], line)\\n        if ignored:\\n            ignored_section = True\\n            line = line[ignored.end():]\\n        elif active:\\n            ignored_section = False\\n            line = line[active.end():]\\n        elif other:\\n            # Unknown headings are not reliably interpreted as current findings.\\n            ignored_section = True\\n            line = line[other.end():]\\n        ambiguous_comma = False\\n        previous_negative = False\\n        parts = re.split(\\"(\\" + cues[\\"clause_separator\\"] + \\")\\", line)\\n        for index, fragment in enumerate(parts):\\n            if index % 2:\\n                ambiguous_comma = fragment == \\",\\" and previous_negative\\n                continue\\n            if fragment.strip():\\n                clause = fragment.strip() + (\\" possible\\" if ambiguous_comma and not re.match(cues[\\"negative\\"], fragment.strip()) else \\"\\")\\n                clauses.append((clause, ignored_section))\\n                previous_negative = bool(re.search(cues[\\"negative\\"], fragment))\\n    return clauses\\n\\n\\ndef _target_contexts(clause: str, frozen: FrozenRuleSet, cues: dict[str, str]) -> list[tuple[TargetRule, str, bool]]:\\n    events = [(match.start(), match.end(), rule) for rule in frozen.rules for match in re.finditer(rule.mention, clause)]\\n    events.extend((match.start(), match.end(), None) for match in re.finditer(cues[\\"other_structure\\"], clause))\\n    events.sort(key=lambda event: (event[0], event[1], event[2].target if event[2] else \\"\\"))\\n    if not events:\\n        return []\\n    inherited_negative = bool(re.search(cues[\\"immediate_negative\\"], clause[:events[0][0]]))\\n    result = []\\n    for index, (start, end, rule) in enumerate(events):\\n        if rule is None:\\n            continue\\n        left, right = 0, len(clause)\\n        preceding_connector = \\"\\"\\n        if index:\\n            previous_end = events[index - 1][1]\\n            connectors = list(re.finditer(cues[\\"coordinate\\"], clause[previous_end:start]))\\n            preceding_connector = connectors[-1].group() if connectors else \\"\\"\\n            if connectors:\\n                offset = connectors[-1].start() if preceding_connector == \\"without\\" else connectors[-1].end()\\n                left = previous_end + offset\\n            else:\\n                left = previous_end\\n        if index + 1 < len(events):\\n            next_start = events[index + 1][0]\\n            connector = re.search(cues[\\"coordinate\\"], clause[end:next_start])\\n            right = end + connector.start() if connector else next_start\\n        context = clause[max(left, start - 80):min(right, end + 80)]\\n        shared_negative = (inherited_negative and preceding_connector not in {\\"with\\", \\"without\\"}\\n                           and not re.search(cues[\\"assertion\\"], context) and not re.search(cues[\\"copula\\"], context))\\n        # A coordinated \'no ACL or MCL tear\' shares the final abnormality.\\n        if shared_negative and rule.category in {\\"structure\\", \\"oa\\"} and not re.search(rule.abnormality, context):\\n            if re.search(rule.abnormality, clause) and re.search(cues[\\"coordinate\\"], clause):\\n                context += \\" \\" + (\\"tear\\" if rule.category == \\"structure\\" else \\"oa\\")\\n        result.append((rule, context, shared_negative))\\n    return result\\n\\n\\ndef _negative_scope(context: str, rule: TargetRule, cues: dict[str, str]) -> tuple[bool, bool]:\\n    \\"\\"\\"Return established absence and ambiguous unrelated negation separately.\\"\\"\\"\\n    match = re.search(rule.mention, context)\\n    if not match:\\n        return False, False\\n    before, after = context[:match.start()], context[match.end():]\\n    established = bool(re.search(cues[\\"immediate_negative\\"], before) or re.search(cues[\\"post_negative\\"], after))\\n    return established, bool(re.search(cues[\\"negative\\"], context) and not established)\\n\\n\\ndef extract_report(report: Any, frozen: FrozenRuleSet, *, max_characters: int = MAX_REPORT_CHARACTERS,\\n                   max_clauses: int = MAX_REPORT_CLAUSES) -> dict[str, Any]:\\n    \\"\\"\\"Extract bounded tri-state candidates without returning input content.\\n\\n    Smaller caller budgets are supported. Larger budgets are rejected so the\\n    canonical prototype remains bounded. Missing, malformed, unsupported and\\n    oversized reports produce UNKNOWN rather than partial-prefix predictions.\\n    \\"\\"\\"\\n    if type(frozen) is not FrozenRuleSet:\\n        raise TypeError(\\"extract_report requires a frozen rule set\\")\\n    frozen.verify()\\n    if type(max_characters) is not int or not 1 <= max_characters <= MAX_REPORT_CHARACTERS:\\n        raise ValueError(\\"invalid report character budget\\")\\n    if type(max_clauses) is not int or not 1 <= max_clauses <= MAX_REPORT_CLAUSES:\\n        raise ValueError(\\"invalid report clause budget\\")\\n    if report is None:\\n        return _unknown_result(frozen, \\"MISSING\\", \\"MISSING_REPORT\\")\\n    if not isinstance(report, str):\\n        return _unknown_result(frozen, \\"INVALID_TYPE\\", \\"INVALID_REPORT_TYPE\\")\\n    if len(report) > max_characters:\\n        return _unknown_result(frozen, \\"CHARACTER_LIMIT\\", \\"CHARACTER_LIMIT\\")\\n    if not report.strip():\\n        return _unknown_result(frozen, \\"EMPTY\\", \\"EMPTY_REPORT\\")\\n    report = unicodedata.normalize(\\"NFKC\\", report).casefold()\\n    if len(report) > max_characters:\\n        return _unknown_result(frozen, \\"CHARACTER_LIMIT\\", \\"CHARACTER_LIMIT\\")\\n    malformed = _unsupported_characters(report)\\n    if malformed:\\n        return _unknown_result(frozen, malformed, malformed)\\n    cues = dict(frozen.cues)\\n    if re.search(cues[\\"nonenglish_cue\\"], report):\\n        return _unknown_result(frozen, \\"UNSUPPORTED_LANGUAGE_CUE\\", \\"UNSUPPORTED_LANGUAGE_CUE\\")\\n    clauses = _clauses(report, cues)\\n    if len(clauses) > max_clauses:\\n        return _unknown_result(frozen, \\"CLAUSE_LIMIT\\", \\"CLAUSE_LIMIT\\")\\n    evidence: dict[str, set[str]] = {target: set() for target in TARGETS}\\n    reasons: dict[str, set[str]] = {target: set() for target in TARGETS}\\n    for clause, ignored in clauses:\\n        for rule, context, inherited_negative in _target_contexts(clause, frozen, cues):\\n            target = rule.target\\n            if ignored or re.search(cues[\\"historical\\"], clause) or re.search(cues[\\"mention_only\\"], clause):\\n                reasons[target].add(\\"NONCURRENT_OR_MENTION_ONLY\\")\\n                continue\\n            if re.search(cues[\\"uncertain\\"], clause):\\n                reasons[target].add(\\"UNCERTAIN_EVIDENCE\\")\\n                continue\\n            abnormal = bool(rule.abnormality and re.search(rule.abnormality, context))\\n            direct_negative, ambiguous_negative = _negative_scope(context, rule, cues)\\n            negative = inherited_negative or direct_negative\\n            if ambiguous_negative and not inherited_negative:\\n                reasons[target].add(\\"AMBIGUOUS_NEGATION_SCOPE\\")\\n                continue\\n            normal = bool(rule.normal and re.search(rule.normal, context) and not re.search(cues[\\"negated_normal\\"], context))\\n            if rule.category == \\"structure\\":\\n                if normal or (negative and (abnormal or re.search(cues[\\"generic_abnormality\\"], context))):\\n                    evidence[target].add(\\"NEGATIVE\\")\\n                if abnormal and not negative:\\n                    evidence[target].add(\\"POSITIVE\\")\\n                if not normal and not abnormal:\\n                    reasons[target].add(\\"MENTION_WITHOUT_EXPLICIT_FINDING\\")\\n            elif rule.category == \\"oa\\":\\n                if abnormal:\\n                    evidence[target].add(\\"NEGATIVE\\" if negative else \\"POSITIVE\\")\\n                else:\\n                    reasons[target].add(\\"COMPARTMENT_WITHOUT_EXPLICIT_OA\\")\\n            else:\\n                if negative:\\n                    evidence[target].add(\\"NEGATIVE\\")\\n                elif target == \\"Contusion\\" and not re.search(cues[\\"bone_contusion\\"], context):\\n                    reasons[target].add(\\"TISSUE_UNSPECIFIED_OR_NONBONE\\")\\n                elif re.search(cues[\\"assertion\\"], context) or re.search(cues[\\"positive_qualifier\\"], context):\\n                    evidence[target].add(\\"POSITIVE\\")\\n                else:\\n                    reasons[target].add(\\"MENTION_WITHOUT_EXPLICIT_FINDING\\")\\n    result: dict[str, Any] = {\\"targets\\": {}, \\"report_status\\": \\"PROCESSED\\", \\"rule_identity\\": frozen.identity()}\\n    for target in TARGETS:\\n        states = evidence[target]\\n        conflict = states == {\\"POSITIVE\\", \\"NEGATIVE\\"}\\n        if conflict:\\n            state = \\"UNKNOWN\\"\\n            reasons[target].add(\\"CONTRADICTORY_EVIDENCE\\")\\n        elif \\"UNCERTAIN_EVIDENCE\\" in reasons[target]:\\n            state = \\"UNKNOWN\\"\\n        elif len(states) == 1:\\n            state = next(iter(states))\\n            reasons[target].add(\\"EXPLICIT_\\" + state)\\n        else:\\n            state = \\"UNKNOWN\\"\\n            if not reasons[target]:\\n                reasons[target].add(\\"NO_TARGET_EVIDENCE\\")\\n        result[\\"targets\\"][target] = {\\"state\\": state, \\"conflict\\": conflict, \\"reason_codes\\": tuple(sorted(reasons[target]))}\\n    return result\\n","submission":"\\"\\"\\"Validate the exact submission contract and write without overwriting files.\\"\\"\\"\\r\\n\\r\\nfrom __future__ import annotations\\r\\n\\r\\nimport csv\\r\\nimport math\\r\\nfrom collections.abc import Iterable, Mapping, Sequence\\r\\nfrom pathlib import Path\\r\\nfrom typing import Any\\r\\n\\r\\nfrom .contracts import ContractError, STUDY_ID, TARGETS, _numeric, validate_identifier\\r\\nfrom .inputs import safe_absolute\\r\\n\\r\\nSUBMISSION_COLUMNS = (STUDY_ID, *TARGETS)\\r\\n\\r\\n\\r\\ndef validate_submission(\\r\\n    rows: Sequence[Mapping[str, Any]],\\r\\n    test_ids: Iterable[str],\\r\\n    columns: Sequence[str] | None = None,\\r\\n) -> dict[str, Any]:\\r\\n    \\"\\"\\"Require each test study once and twelve finite probabilities in order.\\"\\"\\"\\r\\n    expected_ids = [validate_identifier(uid) for uid in test_ids]\\r\\n    if not expected_ids:\\r\\n        raise ContractError(\\"test identifier set must not be empty\\")\\r\\n    if len(set(expected_ids)) != len(expected_ids):\\r\\n        raise ContractError(\\"test identifier list contains duplicates\\")\\r\\n    if not rows:\\r\\n        raise ContractError(\\"submission must contain at least one row\\")\\r\\n    if not isinstance(rows[0], Mapping):\\r\\n        raise ContractError(\\"submission row 0 must be a mapping\\")\\r\\n    actual_columns = tuple(columns) if columns is not None else tuple(rows[0])\\r\\n    if actual_columns != SUBMISSION_COLUMNS:\\r\\n        raise ContractError(\\"submission columns must exactly match the identifier and target order\\")\\r\\n    observed_ids: set[str] = set()\\r\\n    for index, row in enumerate(rows):\\r\\n        if not isinstance(row, Mapping) or set(row) != set(SUBMISSION_COLUMNS):\\r\\n            raise ContractError(f\\"submission row {index} has missing or extra cells\\")\\r\\n        uid = validate_identifier(row[STUDY_ID])\\r\\n        if uid in observed_ids:\\r\\n            raise ContractError(f\\"duplicate submission identifier at row {index}\\")\\r\\n        observed_ids.add(uid)\\r\\n        for target in TARGETS:\\r\\n            value = row[target]\\r\\n            if not _numeric(value):\\r\\n                raise ContractError(f\\"submission row {index} has a nonnumeric probability for {target}\\")\\r\\n            try:\\r\\n                probability = float(value)\\r\\n            except (ValueError, OverflowError) as error:\\r\\n                raise ContractError(f\\"submission row {index} has an invalid probability for {target}\\") from error\\r\\n            if not math.isfinite(probability) or not 0.0 <= probability <= 1.0:\\r\\n                raise ContractError(f\\"submission row {index} has an invalid probability for {target}\\")\\r\\n    if observed_ids != set(expected_ids):\\r\\n        raise ContractError(\\"submission identifier set differs from the exact test identifier set\\")\\r\\n    return {\\r\\n        \\"status\\": \\"PASS\\", \\"row_count\\": len(rows), \\"unique_study_count\\": len(observed_ids),\\r\\n        \\"columns\\": list(SUBMISSION_COLUMNS), \\"probabilities_per_row\\": len(TARGETS),\\r\\n    }\\r\\n\\r\\n\\r\\ndef write_submission(\\r\\n    rows: Sequence[Mapping[str, Any]],\\r\\n    test_ids: Iterable[str],\\r\\n    path: str | Path,\\r\\n    columns: Sequence[str] = SUBMISSION_COLUMNS,\\r\\n) -> dict[str, Any]:\\r\\n    \\"\\"\\"Validate, reject protected/link paths, and refuse any existing output.\\"\\"\\"\\r\\n    validation = validate_submission(rows, test_ids, columns)\\r\\n    destination = safe_absolute(path)\\r\\n    with destination.open(\\"x\\", newline=\\"\\", encoding=\\"utf-8\\") as stream:\\r\\n        writer = csv.DictWriter(stream, fieldnames=list(columns))\\r\\n        writer.writeheader()\\r\\n        writer.writerows(rows)\\r\\n    return {**validation, \\"path\\": str(destination), \\"size_bytes\\": destination.stat().st_size}\\r\\n","validation_support":"\\"\\"\\"Sparse-label support diagnostics, never an approved validation split.\\"\\"\\"\\r\\n\\r\\nfrom __future__ import annotations\\r\\n\\r\\nfrom collections.abc import Mapping\\r\\nfrom typing import Any\\r\\n\\r\\nfrom .contracts import ContractError, STUDY_ID, TARGETS, label_vector, validate_identifier\\r\\n\\r\\n\\r\\nEXACT_SEARCH_MAX_LABELED_STUDIES = {2: 12, 3: 8}\\r\\n\\r\\n\\r\\ndef _joint_coverage(masks: list[int], folds: int) -> dict[str, Any]:\\r\\n    \\"\\"\\"Find only a class-coverage witness on tiny study-grouped problems.\\r\\n\\r\\n    All twenty-four class bits must occur in every fold. No identity, row index\\r\\n    or assignment is returned. Larger problems remain explicitly unassessed.\\r\\n    The first study is fixed to fold zero to remove equivalent fold names.\\r\\n    \\"\\"\\"\\r\\n    limit = EXACT_SEARCH_MAX_LABELED_STUDIES[folds]\\r\\n    base = {\\"scope\\": \\"Observed positive and negative class coverage only; study grouping\\",\\r\\n            \\"labeled_study_count\\": len(masks), \\"exact_search_max_labeled_studies\\": limit,\\r\\n            \\"split_approved\\": False}\\r\\n    if len(masks) > limit:\\r\\n        return {**base, \\"status\\": \\"NOT_ASSESSED\\", \\"reason\\": \\"Tiny exact-search limit exceeded; joint feasibility is not implied by marginal counts\\"}\\r\\n    ordered = sorted(masks, key=lambda value: (-value.bit_count(), value))\\r\\n    complete = (1 << (2 * len(TARGETS))) - 1\\r\\n    suffix_union = [0] * (len(ordered) + 1)\\r\\n    for index in range(len(ordered) - 1, -1, -1):\\r\\n        suffix_union[index] = suffix_union[index + 1] | ordered[index]\\r\\n    visits = 0\\r\\n    failed: set[tuple[int, tuple[int, ...]]] = set()\\r\\n\\r\\n    def search(index: int, covered: tuple[int, ...], counts: tuple[int, ...]) -> tuple[int, ...] | None:\\r\\n        nonlocal visits\\r\\n        visits += 1\\r\\n        if all(value == complete for value in covered):\\r\\n            # Remaining labeled studies may be put in any fold without losing\\r\\n            # coverage. This is not a balanced or patient-safe split proposal.\\r\\n            return (counts[0] + len(ordered) - index, *counts[1:])\\r\\n        if index == len(ordered) or any((value | suffix_union[index]) != complete for value in covered):\\r\\n            return None\\r\\n        key = (index, tuple(sorted(covered)))\\r\\n        if key in failed:\\r\\n            return None\\r\\n        equivalent: set[int] = set()\\r\\n        for fold, value in enumerate(covered):\\r\\n            if value in equivalent:\\r\\n                continue\\r\\n            equivalent.add(value)\\r\\n            updated = list(covered)\\r\\n            updated[fold] |= ordered[index]\\r\\n            sizes = list(counts)\\r\\n            sizes[fold] += 1\\r\\n            result = search(index + 1, tuple(updated), tuple(sizes))\\r\\n            if result is not None:\\r\\n                return result\\r\\n        failed.add(key)\\r\\n        return None\\r\\n\\r\\n    witness = search(1, (ordered[0], *([0] * (folds - 1))), (1, *([0] * (folds - 1)))) if ordered else None\\r\\n    if witness is None:\\r\\n        return {**base, \\"status\\": \\"FAIL\\", \\"reason\\": \\"Joint class coverage is infeasible in the exact study-grouped search\\",\\r\\n                \\"search_states_visited\\": visits}\\r\\n    return {**base, \\"status\\": \\"PASS\\", \\"reason\\": \\"Class-coverage witness found; this is not an approved validation design\\",\\r\\n            \\"search_states_visited\\": visits, \\"witness_labeled_study_counts_per_fold\\": list(witness)}\\r\\n\\r\\n\\r\\ndef _failure(reason: str) -> dict[str, Any]:\\r\\n    return {\\"status\\": \\"FAIL\\", \\"reason\\": reason, \\"training_ready\\": False,\\r\\n            \\"validation_design_status\\": \\"REVIEW_REQUIRED\\", \\"split_approved\\": False}\\r\\n\\r\\n\\r\\ndef summarize_validation(train_rows: list[dict[str, Any]], csv_train_summary: dict[str, Any]) -> dict[str, Any]:\\r\\n    \\"\\"\\"Reuse verified target statistics, preserving every unknown-label mask.\\r\\n\\r\\n    Rechecking counts against unchanged input rows protects this interface from\\r\\n    a stale summary. The optional tiny exact search proves only class coverage,\\r\\n    not balance, reliable estimates, patient safety or a suitable split.\\r\\n    \\"\\"\\"\\r\\n    if not isinstance(csv_train_summary, Mapping) or csv_train_summary.get(\\"status\\") != \\"PASS\\":\\r\\n        return _failure(\\"Valid training CSV contract summary is required\\")\\r\\n    if csv_train_summary.get(\\"row_count\\") != len(train_rows):\\r\\n        return _failure(\\"CSV summary row count does not match supplied training rows\\")\\r\\n    source = csv_train_summary.get(\\"target_statistics\\")\\r\\n    if not isinstance(source, Mapping) or any(target not in source for target in TARGETS):\\r\\n        return _failure(\\"CSV summary must include all twelve verified target statistics\\")\\r\\n    observed = {target: {\\"positive\\": 0, \\"negative\\": 0, \\"observed\\": 0, \\"unknown\\": 0, \\"invalid\\": 0}\\r\\n                for target in TARGETS}\\r\\n    seen: set[str] = set()\\r\\n    masks: list[int] = []\\r\\n    try:\\r\\n        for row in train_rows:\\r\\n            uid = validate_identifier(row[STUDY_ID])\\r\\n            if uid in seen:\\r\\n                return _failure(\\"Study grouping requires unique training study identifiers\\")\\r\\n            seen.add(uid)\\r\\n            vector = label_vector(row)\\r\\n            mask = 0\\r\\n            for index, target in enumerate(TARGETS):\\r\\n                state, value = vector[\\"states\\"][index], vector[\\"values\\"][index]\\r\\n                observed[target][state] += 1\\r\\n                if vector[\\"observed_mask\\"][index]:\\r\\n                    name = \\"positive\\" if value == 1.0 else \\"negative\\"\\r\\n                    observed[target][name] += 1\\r\\n                    mask |= 1 << (2 * index + int(value == 1.0))\\r\\n            if mask:\\r\\n                masks.append(mask)\\r\\n    except (ContractError, KeyError, TypeError):\\r\\n        return _failure(\\"Supplied training rows violate study or label contracts\\")\\r\\n    targets = {}\\r\\n    for target in TARGETS:\\r\\n        stats = source[target]\\r\\n        if not isinstance(stats, Mapping):\\r\\n            return _failure(\\"CSV target statistics are malformed\\")\\r\\n        for key, actual in observed[target].items():\\r\\n            value = stats.get(key)\\r\\n            if type(value) is not int or value < 0 or value != actual:\\r\\n                return _failure(\\"CSV target statistics do not match supplied label observations\\")\\r\\n        if stats[\\"invalid\\"]:\\r\\n            return _failure(\\"Invalid label values cannot establish validation support\\")\\r\\n        # Reuse the verified counts; prevalence uses observed labels only.\\r\\n        targets[target] = {key: stats[key] for key in (\\"positive\\", \\"negative\\", \\"observed\\", \\"unknown\\", \\"invalid\\")}\\r\\n        targets[target][\\"observation_coverage\\"] = stats[\\"observed\\"] / len(train_rows) if train_rows else None\\r\\n        targets[target][\\"support_consequence\\"] = (\\r\\n            \\"Both classes observed; marginal support does not prove joint split feasibility or reliable estimates\\"\\r\\n            if stats[\\"positive\\"] and stats[\\"negative\\"] else\\r\\n            \\"At least one class has no observed support; a two-class validation estimate is unsupported\\"\\r\\n        )\\r\\n    diagnostics = {}\\r\\n    for folds, name in ((2, \\"two_folds\\"), (3, \\"three_folds\\")):\\r\\n        deficient = [target for target in TARGETS if min(targets[target][\\"positive\\"], targets[target][\\"negative\\"]) < folds]\\r\\n        joint = {\\"status\\": \\"FAIL\\", \\"reason\\": \\"Necessary per-class study support is insufficient\\", \\"split_approved\\": False}\\r\\n        if not deficient:\\r\\n            joint = _joint_coverage(masks, folds)\\r\\n        diagnostics[name] = {\\r\\n            \\"fold_count\\": folds, \\"necessary_per_class_support\\": folds,\\r\\n            \\"necessary_support_status\\": \\"FAIL\\" if deficient else \\"PASS\\",\\r\\n            \\"targets_below_necessary_support\\": deficient, \\"joint_class_coverage\\": joint,\\r\\n        }\\r\\n    return {\\r\\n        \\"status\\": \\"PASS\\", \\"source\\": \\"Verified CSV training target_statistics, checked against label_vector observation masks\\",\\r\\n        \\"study_count\\": len(train_rows), \\"labeled_study_count\\": len(masks), \\"targets\\": targets,\\r\\n        \\"observed_label_cells\\": sum(stats[\\"observed\\"] for stats in targets.values()),\\r\\n        \\"unknown_label_cells\\": sum(stats[\\"unknown\\"] for stats in targets.values()),\\r\\n        \\"label_mask_policy\\": \\"Unknown and invalid cells remain unobserved; missing labels are never negative\\",\\r\\n        \\"study_grouped_class_support\\": diagnostics,\\r\\n        \\"patient_grouping\\": {\\"status\\": \\"NOT_ASSESSED\\", \\"reason\\": \\"No trustworthy dataset-wide study-to-patient mapping has been established; sampled PatientID metadata is insufficient\\"},\\r\\n        \\"gold_resource\\": {\\"status\\": \\"NOT_ESTABLISHED\\", \\"reason\\": \\"Observed label counts do not establish an independent 58-study resource or its provenance\\"},\\r\\n        \\"validation_design_status\\": \\"REVIEW_REQUIRED\\", \\"split_approved\\": False, \\"training_ready\\": False,\\r\\n        \\"limitations\\": [\\r\\n            \\"Sparse labels may prevent a reliable twelve-target validation estimate.\\",\\r\\n            \\"Validation design must be reviewed before training.\\",\\r\\n            \\"Necessary marginal positive/negative counts do not guarantee simultaneous class coverage across targets.\\",\\r\\n            \\"A class-coverage witness does not establish balanced folds, patient independence or statistical reliability.\\",\\r\\n            \\"No approved split, patient mapping, independent gold resource or performance estimate is created.\\",\\r\\n        ],\\r\\n    }\\r\\n"}}')
_RSNA_PACKAGE = 'rsna_knee_phase_c_bundle'
for _module, _source in _RSNA_BUNDLE["sources"].items():
    _actual = hashlib.sha256(_source.encode("utf-8")).hexdigest()
    if _actual != _RSNA_BUNDLE["sha256"][_module + ".py"]:
        raise RuntimeError("Embedded source checksum mismatch: " + _module)
_RSNA_SOURCE_IDENTITY = {
    "kind": "embedded-notebook",
    "bundle_sha256": hashlib.sha256(json.dumps(_RSNA_BUNDLE, sort_keys=True, ensure_ascii=True, separators=(",", ":")).encode("utf-8")).hexdigest(),
    "source_sha256": dict(_RSNA_BUNDLE["sha256"]),
}

class _RsnaBundleLoader(importlib.abc.MetaPathFinder, importlib.abc.Loader):
    package_name = _RSNA_PACKAGE

    def __init__(self, sources, fingerprint):
        # Bind this loader to its own bundle, not mutable notebook globals.
        self.sources = dict(sources)
        self.fingerprint = fingerprint

    def find_spec(self, fullname, path=None, target=None):
        if fullname == self.package_name:
            return importlib.util.spec_from_loader(fullname, self, is_package=True)
        prefix = self.package_name + "."
        if fullname.startswith(prefix):
            name = fullname[len(prefix):]
            if name in self.sources and name != "__init__":
                return importlib.util.spec_from_loader(fullname, self)
        return None

    def create_module(self, spec):
        return None

    def exec_module(self, module):
        name = "__init__" if module.__name__ == self.package_name else module.__name__.split(".")[-1]
        module.__file__ = "<embedded:" + name + ".py>"
        module.__rsna_bundle_sha256__ = self.fingerprint
        code = compile(self.sources[name], module.__file__, "exec")
        exec(code, module.__dict__)

# Preserve class identity when re-running an identical, verified bundle.
# Changed sources are reloaded and require the configuration cell to run again.
_rsna_fingerprint = _RSNA_SOURCE_IDENTITY["bundle_sha256"]
_rsna_cached = {
    name: module for name, module in sys.modules.items()
    if name == _RSNA_PACKAGE or name.startswith(_RSNA_PACKAGE + ".")
}
_rsna_same_bundle = (
    _RSNA_PACKAGE in _rsna_cached
    and all(getattr(module, "__rsna_bundle_sha256__", None) == _rsna_fingerprint
            for module in _rsna_cached.values())
)
if not _rsna_same_bundle:
    for _name in _rsna_cached:
        del sys.modules[_name]
    sys.meta_path[:] = [finder for finder in sys.meta_path if getattr(finder, "package_name", None) != _RSNA_PACKAGE]
    sys.meta_path.insert(0, _RsnaBundleLoader(_RSNA_BUNDLE["sources"], _rsna_fingerprint))
from rsna_knee_phase_c_bundle.phase_c_config import PhaseCConfig
from rsna_knee_phase_c_bundle.audit import run_preflight
from rsna_knee_phase_c_bundle.phase_c import run_phase_c, write_phase_c_report


In [ ]:
# Generated limits. Change project configuration and rebuild; no manual cells needed.
CONFIG = {'data_root': None,
 'input_base': '/kaggle/input',
 'inspect_csv': True,
 'max_csv_rows': 100000,
 'max_depth': 2,
 'max_directories': 128,
 'max_entries': 2048,
 'max_extraction_seconds': 120,
 'max_file_bytes': 33554432,
 'max_report_bytes': 1048576,
 'max_report_chars': 20000,
 'max_report_clauses': 256,
 'output_dir': '/kaggle/working/rsna_audit'}
phase_c_config = PhaseCConfig.from_dict(CONFIG)


In [ ]:
if 'phase_c_config' not in globals() or not isinstance(phase_c_config, PhaseCConfig):
    raise RuntimeError('Missing or stale PhaseCConfig: use Run All to recreate the validated configuration.')
csv_preflight = run_preflight(phase_c_config.preflight_config(), source_identity=_RSNA_SOURCE_IDENTITY)
print(json.dumps({'csv_preflight_status': csv_preflight['status'], 'csv_contracts_status': csv_preflight['csv_contracts']['status']}))


In [ ]:
if 'phase_c_config' not in globals() or not isinstance(phase_c_config, PhaseCConfig) or 'csv_preflight' not in globals():
    raise RuntimeError('Run the validated configuration and CSV preflight first, or use Run All.')
if csv_preflight.get('source_identity') != _RSNA_SOURCE_IDENTITY or csv_preflight.get('configuration') != phase_c_config.preflight_config().to_dict():
    raise RuntimeError('The CSV preflight is stale: use Run All before the Phase C audit.')
# Ordered pipeline: report aggregate -> frozen extraction -> observed-label agreement -> feasibility.
phase_c_report = run_phase_c(phase_c_config, source_identity=_RSNA_SOURCE_IDENTITY, preflight=csv_preflight)
print(json.dumps(phase_c_report, sort_keys=True, indent=2, allow_nan=False))


In [ ]:
if 'phase_c_config' not in globals() or not isinstance(phase_c_config, PhaseCConfig) or 'phase_c_report' not in globals():
    raise RuntimeError('Run the configuration and Phase C audit first, or use Run All.')
if phase_c_report.get('source_identity') != _RSNA_SOURCE_IDENTITY or phase_c_report.get('configuration') != phase_c_config.to_dict():
    raise RuntimeError('The Phase C report is stale: use Run All before writing evidence.')
report_path = write_phase_c_report(phase_c_report, phase_c_config)
print(str(report_path))
